# Big Data Analysis of Bangla News with PySpark

**Complete single-notebook project** using the raw Potrika Bengali newspaper corpus.

This notebook performs ingestion, schema and quality auditing, Unicode-aware text cleaning,
date normalization, duplicate handling, temporal and publisher analysis, Bangla token analysis,
TF–IDF, LDA topic discovery, rolling-baseline spike detection, validation, visualisation, and
export of report-ready tables, figures, models, Parquet data, and a final ZIP archive.

The full raw corpus is used for data quality checks and descriptive/trend analysis. The expensive
NLP models use a deterministic category-balanced sample whose size is reported in the outputs.
This keeps the notebook reproducible on a standard Colab runtime without silently replacing the
raw corpus with the pre-balanced dataset.

## 1. Runtime setup

Run the next cell once. In Colab it installs the required packages. In a prepared local runtime,
it skips installation. A runtime restart is normally not required.

In [ ]:
import importlib.util, os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
required = {
    "pyspark": "pyspark==4.0.4",
    "distutils": "setuptools",
    "pyarrow": "pyarrow",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Colab runtime:" if IN_COLAB else "Local verification runtime:", sys.version.split()[0])

## 2. Dataset discovery and project configuration

Supported Colab inputs:

1. Upload `Potrika-Newspaper Datasets in the Bangla Language.zip` to `/content`.
2. Put the extracted `RawDataset` folder in Google Drive and set `POTRIKA_DATA_DIR`.
3. Download the Kaggle archive into `/content` before running this cell.

The cell automatically extracts **only `RawDataset/`** from a discovered archive. To mount Drive,
set `MOUNT_GOOGLE_DRIVE=1` in the first line of the cell before execution.

In [ ]:
import csv
import json
import math
import os
import re
import shutil
import sys
import unicodedata
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


# ============================================================
# BASIC SETTINGS
# ============================================================

SEED = 42


# ============================================================
# GOOGLE COLAB / DRIVE SETUP
# ============================================================

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    work_root = Path("/content")

    # Your actual dataset location
    data_dir = Path("/content/drive/MyDrive/Bangla News/RawDataset")

else:
    work_root = Path.cwd()

    # For running outside Colab
    configured_data = os.environ.get("POTRIKA_DATA_DIR")

    if configured_data:
        data_dir = Path(configured_data)
    else:
        data_dir = work_root / "RawDataset"


# ============================================================
# PROJECT OUTPUT DIRECTORIES
# ============================================================

project_dir = Path(
    os.environ.get(
        "BANGLA_PROJECT_DIR",
        work_root / "bangla_news_project_outputs"
    )
)

table_dir = project_dir / "tables"
fig_dir = project_dir / "figures"
model_dir = project_dir / "models"
parquet_dir = project_dir / "parquet"

for folder in (
    project_dir,
    table_dir,
    fig_dir,
    model_dir,
    parquet_dir
):
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# VERIFY DATASET
# ============================================================

if not data_dir.exists():
    raise FileNotFoundError(
        f"Dataset folder was not found:\n{data_dir}\n\n"
        "Expected Colab location:\n"
        "/content/drive/MyDrive/Bangla News/RawDataset"
    )


# Find all CSV files recursively
csv_files = sorted(data_dir.rglob("*.csv"))

if not csv_files:
    raise RuntimeError(
        f"RawDataset exists, but no CSV files were found under:\n{data_dir}"
    )


# ============================================================
# SUMMARY
# ============================================================

print("=" * 70)
print("DATASET SETUP SUCCESSFUL")
print("=" * 70)

print("Dataset directory:", data_dir)
print("Raw CSV files:", len(csv_files))
print("Project output directory:", project_dir)

print("\nFirst 5 CSV files:")

for f in csv_files[:5]:
    print(" -", f)

## 3. Start Spark and audit source headers

The source-header audit runs before Spark ingestion so schema drift is visible. The canonical
schema is resolved by name, and the source index column is ignored.

In [ ]:
from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.clustering import LDA
from pyspark.ml.stat import Summarizer
from pyspark.ml.functions import vector_to_array

shuffle_partitions = int(os.environ.get("SPARK_SHUFFLE_PARTITIONS", "64"))
driver_memory = os.environ.get("SPARK_DRIVER_MEMORY", "5g")
master = os.environ.get("SPARK_MASTER", "local[*]")

spark = (
    SparkSession.builder.master(master)
    .appName("BanglaNewsBigDataCompleteProject")
    .config("spark.sql.shuffle.partitions", str(shuffle_partitions))
    .config("spark.default.parallelism", str(shuffle_partitions))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.driver.memory", driver_memory)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)

def normalise_header(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())

header_rows = []
for path in csv_files:
    with path.open("r", encoding="utf-8-sig", errors="replace", newline="") as handle:
        header = next(csv.reader(handle))
    header_rows.append({"file": str(path.relative_to(data_dir)), "header": " | ".join(header)})
header_audit_pd = pd.DataFrame(header_rows)
print("Distinct header layouts:", header_audit_pd["header"].nunique())
display(header_audit_pd.drop_duplicates("header"))

## 4. Ingest and standardise all raw CSV files

In [ ]:
raw_input = (
    spark.read.option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("encoding", "UTF-8")
    .csv([str(p) for p in csv_files])
    .withColumn("_file_path", F.input_file_name())
)

aliases = {
    "news_article": {"news", "newsarticle", "article", "content", "body", "articletext"},
    "category_raw": {"category", "class", "label", "topic", "newscategory"},
    "headline": {"heading", "headline", "title", "newstitle"},
    "publication_date_raw": {"date", "publicationdate", "publishdate", "publisheddate"},
    "newspaper_source_raw": {"source", "newspaper", "paper", "portal", "newspapersource"},
}
norm_to_original = {normalise_header(c): c for c in raw_input.columns}
selected = []
for canonical, names in aliases.items():
    source_col = next((norm_to_original[n] for n in names if n in norm_to_original), None)
    selected.append(
        F.col(source_col).cast("string").alias(canonical)
        if source_col else F.lit(None).cast("string").alias(canonical)
    )
raw_df = raw_input.select(*selected, "_file_path").persist(StorageLevel.DISK_ONLY)
raw_count = raw_df.count()
print(f"Raw rows: {raw_count:,}")
raw_df.printSchema()
display(raw_df.limit(5).toPandas())

## 5. Data quality audit, label harmonisation, Unicode cleaning, and date parsing

In [ ]:
# ============================================================
# DATA CLEANING, STANDARDIZATION, AND QUALITY AUDIT
# ============================================================

audit_cols = [
    "news_article",
    "category_raw",
    "headline",
    "publication_date_raw",
    "newspaper_source_raw"
]


# ============================================================
# 1. MISSING / BLANK VALUE AUDIT BEFORE CLEANING
# ============================================================

missing_before = raw_df.agg(*[
    F.sum(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in audit_cols
]).toPandas().T.reset_index()

missing_before.columns = ["column", "missing_or_blank"]

print("=" * 70)
print("MISSING / BLANK VALUES BEFORE CLEANING")
print("=" * 70)

display(missing_before)


# ============================================================
# 2. CATEGORY STANDARDIZATION
# ============================================================

path_lower = F.lower(F.col("_file_path"))

# If category is unavailable in the CSV, try extracting it
# from the parent directory in the file path.
path_category = F.regexp_extract(
    F.col("_file_path"),
    r"/([^/]+)/[^/]+\.csv$",
    1
)

category_seed = F.coalesce(
    F.nullif(F.trim(F.col("category_raw")), F.lit("")),
    path_category
)

# Create simplified lowercase key for matching.
category_key = F.regexp_replace(
    F.lower(category_seed),
    r"[^a-z]+",
    ""
)

category = (
    F.when(
        category_key.contains("science") |
        category_key.contains("technology"),
        "Science & Technology"
    )
    .when(
        category_key.contains("world") |
        category_key.contains("international"),
        "International"
    )
    .when(
        category_key.contains("national"),
        "National"
    )
    .when(
        category_key.contains("sport"),
        "Sports"
    )
    .when(
        category_key.contains("entertainment"),
        "Entertainment"
    )
    .when(
        category_key.contains("econom"),
        "Economy"
    )
    .when(
        category_key.contains("education"),
        "Education"
    )
    .when(
        category_key.contains("politic"),
        "Politics"
    )
    .otherwise(
        F.initcap(category_seed)
    )
)


# ============================================================
# 3. NEWSPAPER SOURCE STANDARDIZATION
# ============================================================

source_seed = F.coalesce(
    F.nullif(
        F.trim(F.col("newspaper_source_raw")),
        F.lit("")
    ),
    path_lower
)

source_key = F.regexp_replace(
    F.lower(source_seed),
    r"[^a-z]+",
    ""
)

source = (
    F.when(
        source_key.contains("jugantor"),
        "Jugantor"
    )
    .when(
        source_key.contains("jaijaidin"),
        "Jaijaidin"
    )
    .when(
        source_key.contains("ittefaq"),
        "Ittefaq"
    )
    .when(
        source_key.contains("kaler"),
        "Kaler Kontho"
    )
    .when(
        source_key.contains("inqilab"),
        "Inqilab"
    )
    .when(
        source_key.contains("somoyer"),
        "Somoyer Alo"
    )
    .otherwise(
        F.initcap(
            F.regexp_replace(
                source_seed,
                "_",
                " "
            )
        )
    )
)


# ============================================================
# 4. OPTIONAL BANGLA UNICODE NFC NORMALIZATION
# ============================================================

# Spark does not provide a convenient native NFC normalization
# function here.
#
# The supplied corpus is already expected to be canonically
# encoded, so this is OFF by default to avoid sending several
# GB of text through a Python UDF.

ENABLE_NFC_UDF = (
    os.environ.get("ENABLE_NFC_UDF", "0") == "1"
)


@F.udf(T.StringType())
def normalise_nfc_udf(value):
    if isinstance(value, str):
        return unicodedata.normalize("NFC", value)
    return value


# ============================================================
# 5. TEXT CLEANING FUNCTION
# ============================================================

def clean_text(column_name):

    if ENABLE_NFC_UDF:
        c = normalise_nfc_udf(F.col(column_name))
    else:
        c = F.col(column_name)

    # Remove URLs
    c = F.regexp_replace(
        c,
        r"https?://\S+|www\.\S+",
        " "
    )

    # Remove HTML tags
    c = F.regexp_replace(
        c,
        r"<[^>]+>",
        " "
    )

    # Remove zero-width / invisible Unicode characters
    c = F.regexp_replace(
        c,
        r"[\u200B-\u200D\uFEFF\u00A0]",
        " "
    )

    # Keep:
    # Bangla characters
    # English letters
    # numbers
    # whitespace
    c = F.regexp_replace(
        c,
        r"[^\u0980-\u09FFA-Za-z0-9\s]",
        " "
    )

    # Collapse repeated whitespace
    c = F.regexp_replace(
        c,
        r"\s+",
        " "
    )

    return F.trim(c)


# ============================================================
# 6. ROBUST PUBLICATION DATE PARSING
# ============================================================

# IMPORTANT:
# Some raw records contain impossible calendar dates such as:
#
#     2019/02/30
#
# Standard to_date() can raise an exception under Spark ANSI
# mode. try_to_timestamp() safely converts impossible dates
# to NULL instead of crashing the pipeline.
#
# We DO NOT manually repair impossible dates because their
# true publication date cannot be known reliably.


parsed_date = F.coalesce(

    # --------------------------------------------------------
    # Format 1: yyyy/MM/dd
    # Example: 2019/02/28
    # --------------------------------------------------------

    F.expr("""
        CAST(
            try_to_timestamp(
                trim(publication_date_raw),
                'yyyy/MM/dd'
            )
            AS DATE
        )
    """),

    # --------------------------------------------------------
    # Format 2: yyyy-MM-dd
    # Example: 2019-02-28
    # --------------------------------------------------------

    F.expr("""
        CAST(
            try_to_timestamp(
                trim(publication_date_raw),
                'yyyy-MM-dd'
            )
            AS DATE
        )
    """),

    # --------------------------------------------------------
    # Format 3: dd/MM/yyyy
    # Example: 28/02/2019
    # --------------------------------------------------------

    F.expr("""
        CAST(
            try_to_timestamp(
                trim(publication_date_raw),
                'dd/MM/yyyy'
            )
            AS DATE
        )
    """),

    # --------------------------------------------------------
    # Format 4: dd-MM-yyyy
    # Example: 28-02-2019
    # --------------------------------------------------------

    F.expr("""
        CAST(
            try_to_timestamp(
                trim(publication_date_raw),
                'dd-MM-yyyy'
            )
            AS DATE
        )
    """)
)


# ============================================================
# 7. CREATE CLEANED / PREPARED DATAFRAME
# ============================================================

prepared_df = (
    raw_df

    # Standardized category
    .withColumn(
        "category",
        category
    )

    # Standardized newspaper source
    .withColumn(
        "newspaper_source",
        source
    )

    # Clean article text
    .withColumn(
        "news_article_clean",
        clean_text("news_article")
    )

    # Clean headline
    .withColumn(
        "headline_clean",
        clean_text("headline")
    )

    # Safely parsed publication date
    .withColumn(
        "publication_date",
        parsed_date
    )

    .select(
        "news_article",
        "news_article_clean",
        "headline",
        "headline_clean",
        "category",
        "newspaper_source",
        "publication_date_raw",
        "publication_date",
        "_file_path"
    )

    # Dataset is large, so avoid unnecessary memory pressure.
    .persist(StorageLevel.DISK_ONLY)
)


# ============================================================
# 8. DATE QUALITY AUDIT
# ============================================================

print("\n" + "=" * 70)
print("PUBLICATION DATE QUALITY AUDIT")
print("=" * 70)


# Overall date statistics
date_audit_pd = (
    prepared_df
    .agg(

        F.count("*").alias("rows"),

        F.sum(
            F.col("publication_date")
            .isNull()
            .cast("long")
        ).alias("missing_or_invalid_dates"),

        F.min(
            "publication_date"
        ).alias("min_date"),

        F.max(
            "publication_date"
        ).alias("max_date")

    )
    .toPandas()
)

display(date_audit_pd)


# ============================================================
# 9. IDENTIFY INVALID RAW DATE VALUES
# ============================================================

# These are values that existed in the raw dataset but could
# not be converted into a valid calendar date.

invalid_dates_df = (
    prepared_df

    .filter(
        F.col("publication_date").isNull()
        &
        F.col("publication_date_raw").isNotNull()
        &
        (
            F.trim(
                F.col("publication_date_raw")
            ) != ""
        )
    )

    .groupBy(
        "publication_date_raw"
    )

    .count()

    .orderBy(
        F.desc("count")
    )
)


invalid_date_rows = (
    prepared_df

    .filter(
        F.col("publication_date").isNull()
    )

    .count()
)


print(
    "\nRows with missing or invalid publication dates:",
    invalid_date_rows
)


print("\nMost common invalid raw date values:")

invalid_dates_df.show(
    30,
    truncate=False
)


# ============================================================
# 10. CATEGORY DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("STANDARDIZED CATEGORY DISTRIBUTION")
print("=" * 70)

(
    prepared_df
    .groupBy("category")
    .count()
    .orderBy(
        F.desc("count")
    )
    .show(
        20,
        truncate=False
    )
)


# ============================================================
# 11. NEWSPAPER SOURCE DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("STANDARDIZED NEWSPAPER SOURCE DISTRIBUTION")
print("=" * 70)

(
    prepared_df
    .groupBy("newspaper_source")
    .count()
    .orderBy(
        F.desc("count")
    )
    .show(
        20,
        truncate=False
    )
)


# ============================================================
# 12. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("PREPROCESSING AND DATA QUALITY AUDIT COMPLETE")
print("=" * 70)

print(
    "\nInvalid calendar dates are retained in "
    "'publication_date_raw' for auditing but represented as "
    "NULL in 'publication_date'."
)

print(
    "They can therefore be excluded safely from subsequent "
    "date-dependent temporal and spike analyses."
)

## 6. Duplicate audit, cleaned dataset, and engineered features

Exact duplicates are defined by cleaned article text, date, and newspaper. Rows with missing dates
remain available for category/source/text analysis but are excluded from date-based calculations.

In [ ]:
exact_dup_groups = (
    prepared_df.filter(F.length("news_article_clean") > 20)
    .groupBy("news_article_clean", "publication_date", "newspaper_source")
    .count().filter(F.col("count") > 1)
)
duplicate_extra_rows = exact_dup_groups.agg(
    F.coalesce(F.sum(F.col("count") - 1), F.lit(0)).alias("extra")
).first()["extra"]

clean_df = (
    prepared_df.filter(F.length("news_article_clean") > 20)
    .filter(F.col("category").isNotNull())
    .dropDuplicates(["news_article_clean", "publication_date", "newspaper_source"])
)
feature_df = (
    clean_df.withColumn("year", F.year("publication_date"))
    .withColumn("month", F.month("publication_date"))
    .withColumn("quarter", F.quarter("publication_date"))
    .withColumn("day_of_week", F.date_format("publication_date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("publication_date").isin([6, 7]).cast("int"))
    .withColumn("article_char_count", F.length("news_article_clean"))
    .withColumn("word_count", F.size(F.split(F.trim("news_article_clean"), r"\s+")))
    .withColumn(
        "season",
        F.when(F.col("month").isin([4, 5]), "Grishma")
        .when(F.col("month").isin([6, 7]), "Barsha")
        .when(F.col("month").isin([8, 9]), "Sharat")
        .when(F.col("month").isin([10, 11]), "Hemanta")
        .when(F.col("month").isin([12, 1]), "Sheet")
        .when(F.col("month").isin([2, 3]), "Basanta")
    )
    .persist(StorageLevel.DISK_ONLY)
)
clean_count = feature_df.count()
print(f"Exact duplicate extra rows: {duplicate_extra_rows:,}")
print(f"Clean rows: {clean_count:,}")
print(f"Rows removed: {raw_count - clean_count:,}")

validation_pd = feature_df.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("publication_date").isNull().cast("long")).alias("missing_dates"),
    F.sum(F.col("newspaper_source").isNull().cast("long")).alias("missing_sources"),
    F.round(F.avg("word_count"), 2).alias("avg_words"),
    F.expr("percentile_approx(word_count, 0.5)").alias("median_words"),
    F.min("publication_date").alias("min_date"),
    F.max("publication_date").alias("max_date"),
).toPandas()
display(validation_pd)

## 7. Save the cleaned Spark dataset as partitioned Parquet

In [ ]:
cleaned_parquet = parquet_dir / "potrika_cleaned"
try:
    (feature_df.write.mode("overwrite").partitionBy("year").parquet(str(cleaned_parquet)))
    parquet_status = "saved"
except Exception as exc:
    parquet_status = f"not saved in this runtime ({type(exc).__name__}); Colab/Linux export remains enabled"
print("Parquet:", parquet_status)

# Part A — Exploratory analysis and category trends

In [ ]:
feature_df.createOrReplaceTempView("bangla_news")

category_summary = feature_df.groupBy("category").count().withColumn(
    "percentage", F.round(F.col("count") * 100 / F.sum("count").over(Window.partitionBy()), 2)
).orderBy(F.desc("count"))
source_summary = feature_df.groupBy("newspaper_source").count().withColumn(
    "percentage", F.round(F.col("count") * 100 / F.sum("count").over(Window.partitionBy()), 2)
).orderBy(F.desc("count"))
yearly = feature_df.filter("year is not null").groupBy("year").count().orderBy("year")
monthly = feature_df.filter("publication_date is not null").groupBy("year", "month").count().orderBy("year", "month")
seasonal = feature_df.filter("season is not null").groupBy("season").count().orderBy(F.desc("count"))
weekpart = feature_df.filter("publication_date is not null").withColumn(
    "weekpart", F.when(F.col("is_weekend") == 1, "Weekend").otherwise("Weekday")
).groupBy("weekpart").count()

year_category = feature_df.filter("year is not null").groupBy("year", "category").count()
year_totals = yearly.withColumnRenamed("count", "year_total")
year_category_share = year_category.join(year_totals, "year").withColumn(
    "category_share_pct", F.round(F.col("count") * 100 / F.col("year_total"), 3)
).orderBy("year", "category")

w_cat = Window.partitionBy("category").orderBy("year")
category_growth = year_category.withColumn("previous_year_count", F.lag("count").over(w_cat)).withColumn(
    "growth_pct", F.when(
        F.col("previous_year_count") > 0,
        F.round((F.col("count") - F.col("previous_year_count")) * 100 / F.col("previous_year_count"), 2)
    )
).orderBy("category", "year")

monthly_category = feature_df.filter("publication_date is not null").withColumn(
    "year_month", F.date_trunc("month", "publication_date")
).groupBy("year_month", "category").count()
w_month = Window.partitionBy("category").orderBy("year_month").rowsBetween(-2, 0)
monthly_category_roll = monthly_category.withColumn(
    "rolling_3m_avg", F.round(F.avg("count").over(w_month), 2)
).orderBy("category", "year_month")

display(category_summary.toPandas())
display(source_summary.toPandas())
display(yearly.toPandas())

# Part B — Publisher comparison and article-length analysis

In [ ]:
source_category = feature_df.groupBy("newspaper_source", "category").count()
w_source = Window.partitionBy("newspaper_source")
source_category_share = source_category.withColumn(
    "source_category_pct", F.round(F.col("count") * 100 / F.sum("count").over(w_source), 2)
).orderBy("newspaper_source", F.desc("count"))

publisher_summary = feature_df.groupBy("newspaper_source").agg(
    F.count("*").alias("article_count"),
    F.round(F.avg("word_count"), 2).alias("avg_word_count"),
    F.countDistinct("publication_date").alias("active_days"),
    F.min("publication_date").alias("first_date"),
    F.max("publication_date").alias("last_date"),
).withColumn(
    "articles_per_active_day",
    F.when(F.col("active_days") > 0, F.round(F.col("article_count") / F.col("active_days"), 2))
).orderBy(F.desc("article_count"))

quantiles = feature_df.approxQuantile("word_count", [0.01, .10, .25, .50, .75, .90, .99], 0.01)
quantile_table = pd.DataFrame({
    "quantile": ["1%", "10%", "25%", "50%", "75%", "90%", "99%"],
    "word_count": quantiles,
})
display(publisher_summary.toPandas())
display(quantile_table)

# Part C — Category-balanced NLP sample and Bangla token analysis

`NLP_MAX_DOCS` limits only the computationally expensive NLP stages. All earlier results use the
complete cleaned raw corpus. Sampling fractions are calculated separately by category.

In [ ]:
NLP_MAX_DOCS = int(os.environ.get("NLP_MAX_DOCS", "120000"))
category_counts = {r["category"]: r["count"] for r in category_summary.collect()}
per_category_target = max(1, NLP_MAX_DOCS // max(1, len(category_counts)))
fractions = {k: min(1.0, per_category_target / v) for k, v in category_counts.items() if v > 0}
nlp_base_df = feature_df.sampleBy("category", fractions=fractions, seed=SEED).persist(StorageLevel.DISK_ONLY)

BANGLA_STOPWORDS = [
    "আমি","আমরা","তুমি","তোমরা","সে","তিনি","তারা","এ","এই","ঐ","ও","ওই","এক","একটি","একটা",
    "কোন","কোনো","কিছু","সব","সকল","অনেক","আর","এবং","অথবা","কিন্তু","তবে","যে","যা","যার",
    "যাদের","যাকে","যেখানে","যখন","যদি","তাহলে","না","নয়","নেই","হয়","হয়","হয়ে","হয়ে","হতে",
    "হবে","হয়েছে","হয়েছে","ছিল","ছিলেন","আছে","আছেন","থেকে","পর্যন্ত","জন্য","দিকে","উপর","নিচে",
    "মধ্যে","সঙ্গে","সাথে","কাছে","পরে","আগে","প্রতি","দিয়ে","দিয়ে","নিয়ে","নিয়ে","করে","করেন",
    "করেছে","করা","করতে","কি","কী","কেন","কিভাবে","যেমন","তেমন","এমন","এখন","তখন","আজ","গত",
    "নতুন","বলেন","বলে","বলা","হিসেবে","মতো","এর","এদের","এরপর","এতে","এটি","এটা","তার","তাঁর",
    "তাদের","নিজের","নিজে","দুই","তিন","চার","আরও","খুব","শুধু","প্রায়","প্রায়","এছাড়া","এছাড়া",
    "উল্লেখ","জানান","জানিয়েছে","জানিয়েছে","জানা","দেন","দেওয়া","দেওয়া"
]
tokenizer = RegexTokenizer(
    inputCol="news_article_clean", outputCol="tokens", pattern=r"[^\u0980-\u09FFA-Za-z0-9]+",
    gaps=True, minTokenLength=2, toLowercase=True
)
remover = StopWordsRemover(
    inputCol="tokens", outputCol="tokens_filtered", stopWords=BANGLA_STOPWORDS, caseSensitive=False
)
tokenized_df = remover.transform(tokenizer.transform(nlp_base_df)).persist(StorageLevel.DISK_ONLY)
nlp_count = tokenized_df.count()
print(f"NLP sample rows: {nlp_count:,} ({nlp_count / clean_count:.1%} of clean corpus)")

word_freq = tokenized_df.select(F.explode("tokens_filtered").alias("term")).groupBy("term").count().orderBy(F.desc("count"))
category_term_freq = tokenized_df.select("category", F.explode("tokens_filtered").alias("term")).groupBy("category", "term").count()
top_terms_by_category = category_term_freq.withColumn(
    "rank", F.row_number().over(Window.partitionBy("category").orderBy(F.desc("count"), "term"))
).filter(F.col("rank") <= 20).orderBy("category", "rank")
display(word_freq.limit(30).toPandas())
display(top_terms_by_category.toPandas().head(80))

# Part D — TF–IDF and keyword evolution

In [ ]:
cv = CountVectorizer(inputCol="tokens_filtered", outputCol="tf", vocabSize=15000, minDF=8.0)
cv_model = cv.fit(tokenized_df)
tf_df = cv_model.transform(tokenized_df)
idf_model = IDF(inputCol="tf", outputCol="tfidf", minDocFreq=5).fit(tf_df)
tfidf_df = idf_model.transform(tf_df).persist(StorageLevel.DISK_ONLY)
vocabulary = cv_model.vocabulary

mean_vectors = tfidf_df.groupBy("category").agg(Summarizer.mean(F.col("tfidf")).alias("mean_tfidf")).collect()
tfidf_rows = []
for row in mean_vectors:
    values = row["mean_tfidf"].toArray()
    for rank, idx in enumerate(np.argsort(values)[::-1][:20], 1):
        tfidf_rows.append({
            "category": row["category"], "rank": rank,
            "term": vocabulary[int(idx)], "mean_tfidf": float(values[int(idx)])
        })
tfidf_keywords_pd = pd.DataFrame(tfidf_rows)

TRACK_KEYWORDS = ["বাংলাদেশ", "সরকার", "নির্বাচন", "শিক্ষা", "অর্থনীতি", "খেলা", "প্রযুক্তি", "করোনা"]
exploded_year_terms = tokenized_df.filter("year is not null").select("year", F.explode("tokens_filtered").alias("term"))
year_token_totals = exploded_year_terms.groupBy("year").count().withColumnRenamed("count", "year_total_tokens")
keyword_evolution = exploded_year_terms.filter(F.col("term").isin(TRACK_KEYWORDS)).groupBy("year", "term").count().join(
    year_token_totals, "year"
).withColumn("per_100k_tokens", F.round(F.col("count") * 100000 / F.col("year_total_tokens"), 3)).orderBy("year", "term")
display(tfidf_keywords_pd.head(60))
display(keyword_evolution.toPandas().head(80))

# Part E — LDA topic discovery and model comparison

The notebook compares multiple topic counts with log perplexity and selects the lowest-perplexity
model. Topic numbers are statistical identifiers; the exported top-word table should be used when
assigning human-readable labels.

In [ ]:
LDA_K_VALUES = [int(v) for v in os.environ.get("LDA_K_VALUES", "5,8,10").split(",")]
LDA_MAX_ITER = int(os.environ.get("LDA_MAX_ITER", "10"))
lda_cv_model = CountVectorizer(
    inputCol="tokens_filtered", outputCol="lda_features", vocabSize=6000, minDF=10.0
).fit(tokenized_df)
lda_features_all = lda_cv_model.transform(tokenized_df).select(
    "category", "year", "publication_date", "lda_features"
).persist(StorageLevel.DISK_ONLY)
lda_train = lda_features_all.sample(False, 0.75, SEED).select("lda_features").persist(StorageLevel.DISK_ONLY)
lda_train_count = lda_train.count()
print(f"LDA training rows: {lda_train_count:,}")

lda_models, lda_eval_rows = {}, []
for k in LDA_K_VALUES:
    model = LDA(
        k=k, maxIter=LDA_MAX_ITER, featuresCol="lda_features", seed=SEED, optimizer="online"
    ).fit(lda_train)
    lda_models[k] = model
    lda_eval_rows.append({
        "k": k,
        "log_likelihood": float(model.logLikelihood(lda_train)),
        "log_perplexity": float(model.logPerplexity(lda_train)),
    })
    print("Finished k =", k)
lda_eval_pd = pd.DataFrame(lda_eval_rows).sort_values("k")
BEST_K = int(lda_eval_pd.sort_values(["log_perplexity", "k"]).iloc[0]["k"])
lda_model = lda_models[BEST_K]
display(lda_eval_pd)
print("Selected topic count:", BEST_K)

In [ ]:
lda_vocab = lda_cv_model.vocabulary
topic_term_rows = []
for row in lda_model.describeTopics(maxTermsPerTopic=12).collect():
    for rank, (idx, weight) in enumerate(zip(row["termIndices"], row["termWeights"]), 1):
        topic_term_rows.append({
            "topic": int(row["topic"]), "rank": rank,
            "term": lda_vocab[int(idx)], "weight": float(weight)
        })
topic_terms_pd = pd.DataFrame(topic_term_rows)
display(topic_terms_pd)

topic_doc_df = lda_model.transform(lda_features_all).withColumn(
    "topic_array", vector_to_array("topicDistribution")
).withColumn("max_topic_prob", F.array_max("topic_array")).withColumn(
    "dominant_topic", F.expr("array_position(topic_array, array_max(topic_array)) - 1")
).persist(StorageLevel.DISK_ONLY)

topic_by_category = topic_doc_df.groupBy("category", "dominant_topic").count().withColumn(
    "within_category_pct", F.round(F.col("count") * 100 / F.sum("count").over(Window.partitionBy("category")), 2)
).orderBy("category", "dominant_topic")
topic_by_year = topic_doc_df.filter("year is not null").groupBy("year", "dominant_topic").count().withColumn(
    "year_topic_pct", F.round(F.col("count") * 100 / F.sum("count").over(Window.partitionBy("year")), 2)
).orderBy("year", "dominant_topic")
display(topic_by_category.toPandas().head(80))

# Part F — Daily rolling-baseline spike detection

A candidate spike requires a 28-day historical baseline, at least 14 prior observations, and a
z-score of at least 3. Spikes are descriptive anomalies and are not treated as proof of real-world causation.

In [ ]:
# Enhanced spike attribution: category, source, and lexical keyness

# Overall spike detector (same 28-day historical baseline as before).
bounds = feature_df.filter("publication_date is not null").agg(
    F.min("publication_date").alias("min_date"), F.max("publication_date").alias("max_date")
).first()
min_date, max_date = bounds["min_date"], bounds["max_date"]
calendar_df = spark.sql(
    f"SELECT explode(sequence(to_date('{min_date}'), to_date('{max_date}'), interval 1 day)) AS publication_date"
)
daily = calendar_df.join(
    feature_df.filter("publication_date is not null").groupBy("publication_date").count().withColumnRenamed("count", "daily_articles"),
    "publication_date", "left"
).fillna({"daily_articles": 0}).orderBy("publication_date")
rolling_w = Window.orderBy("publication_date").rowsBetween(-28, -1)
daily_scored = daily.withColumn("baseline_n", F.count("daily_articles").over(rolling_w)).withColumn(
    "rolling_mean_28", F.avg("daily_articles").over(rolling_w)
).withColumn("rolling_std_28", F.stddev_samp("daily_articles").over(rolling_w)).withColumn(
    "z_score", F.when(
        (F.col("rolling_std_28") > 0) & (F.col("baseline_n") >= 14),
        (F.col("daily_articles") - F.col("rolling_mean_28")) / F.col("rolling_std_28")
    )
).withColumn("is_spike", (
    (F.col("baseline_n") >= 14) & (F.col("rolling_std_28") > 0) & (F.col("z_score") >= 3.0)
).cast("int"))
spikes = daily_scored.filter(F.col("is_spike") == 1).orderBy(F.desc("z_score"))
spike_dates = spikes.select("publication_date").distinct()

# Helper: score each category/source against ITS OWN preceding 28 calendar days.
def dimension_spike_attribution(dimension, value_col, top_n=3):
    values = feature_df.filter(F.col(dimension).isNotNull()).select(dimension).distinct()
    grid = calendar_df.crossJoin(values)
    counts = feature_df.filter("publication_date is not null").filter(F.col(dimension).isNotNull()).groupBy(
        "publication_date", dimension
    ).count().withColumnRenamed("count", value_col)
    series = grid.join(counts, ["publication_date", dimension], "left").fillna({value_col: 0})
    w = Window.partitionBy(dimension).orderBy("publication_date").rowsBetween(-28, -1)
    scored = series.withColumn("baseline_n", F.count(value_col).over(w)).withColumn(
        "baseline_mean_28", F.avg(value_col).over(w)
    ).withColumn("baseline_std_28", F.stddev_samp(value_col).over(w)).withColumn(
        "dimension_z", F.when(
            (F.col("baseline_n") >= 14) & (F.col("baseline_std_28") > 0),
            (F.col(value_col) - F.col("baseline_mean_28")) / F.col("baseline_std_28")
        )
    )
    on_spikes = scored.join(spike_dates, "publication_date", "inner")
    date_totals = feature_df.join(spike_dates, "publication_date").groupBy("publication_date").count().withColumnRenamed("count", "spike_day_total")
    ranked = on_spikes.join(date_totals, "publication_date", "left").withColumn(
        "day_share_pct", F.round(100 * F.col(value_col) / F.col("spike_day_total"), 2)
    ).withColumn(
        "rank", F.row_number().over(Window.partitionBy("publication_date").orderBy(F.desc_nulls_last("dimension_z"), F.desc(value_col), dimension))
    )
    return scored, ranked.filter(F.col("rank") <= top_n).orderBy("publication_date", "rank")

category_daily_scored, spike_category_attribution = dimension_spike_attribution("category", "category_articles", 3)
source_daily_scored, spike_source_attribution = dimension_spike_attribution("newspaper_source", "source_articles", 5)

# Cross-source agreement: fraction of active sources whose own z-score is elevated (>= 2).
source_on_spikes = source_daily_scored.join(spike_dates, "publication_date", "inner")
source_agreement = source_on_spikes.groupBy("publication_date").agg(
    F.sum((F.col("source_articles") > 0).cast("int")).alias("active_sources"),
    F.sum(((F.col("source_articles") > 0) & (F.col("dimension_z") >= 2.0)).cast("int")).alias("elevated_sources")
).withColumn(
    "source_agreement_pct", F.round(100 * F.col("elevated_sources") / F.col("active_sources"), 2)
)

# Lexical keyness, restricted to spike windows BEFORE tokenizing (not after).
# Tokenizing the full corpus first means exploding ~660K articles into tokens before the
# join even runs. Instead: join spike windows against the corpus at the ARTICLE level first
# (a cheap date-range join, no token explosion), keep only articles inside some spike's own
# 28-day window (including the spike day itself), and tokenize only that smaller subset.
# Dunning log-likelihood G^2 then compares spike-day tokens with the preceding 28-day baseline.
spike_window_bounds = spike_dates.select(
    F.col("publication_date").alias("spike_date"),
    F.date_sub(F.col("publication_date"), 28).alias("window_start"),
)
windowed_articles = feature_df.filter("publication_date is not null").select(
    "publication_date", "news_article_clean"
).join(
    spike_window_bounds,
    (F.col("publication_date") >= F.col("window_start")) & (F.col("publication_date") <= F.col("spike_date")),
    "inner",
).select("spike_date", "publication_date", "news_article_clean")

windowed_tokens = remover.transform(tokenizer.transform(windowed_articles)).select(
    "spike_date", "publication_date", F.explode("tokens_filtered").alias("term")
).filter(F.length("term") >= 2).persist(StorageLevel.DISK_ONLY)

term_counts = windowed_tokens.groupBy("spike_date", "term").agg(
    F.sum((F.col("publication_date") == F.col("spike_date")).cast("long")).alias("spike_tf"),
    F.sum((F.col("publication_date") < F.col("spike_date")).cast("long")).alias("baseline_tf")
)
token_totals = windowed_tokens.groupBy("spike_date").agg(
    F.sum((F.col("publication_date") == F.col("spike_date")).cast("long")).alias("spike_tokens"),
    F.sum((F.col("publication_date") < F.col("spike_date")).cast("long")).alias("baseline_tokens")
)
keyness = term_counts.join(token_totals, "spike_date").filter(
    (F.col("spike_tf") >= 3) & (F.col("spike_tokens") > 0) & (F.col("baseline_tokens") > 0)
).withColumn("a", F.col("spike_tf").cast("double")).withColumn(
    "b", F.col("baseline_tf").cast("double")
).withColumn("c", F.col("spike_tokens") - F.col("spike_tf")).withColumn(
    "d", F.col("baseline_tokens") - F.col("baseline_tf")
).withColumn("N", F.col("spike_tokens") + F.col("baseline_tokens")).withColumn(
    "e_a", (F.col("a") + F.col("b")) * F.col("spike_tokens") / F.col("N")
).withColumn("e_b", (F.col("a") + F.col("b")) * F.col("baseline_tokens") / F.col("N")).withColumn(
    "e_c", (F.col("c") + F.col("d")) * F.col("spike_tokens") / F.col("N")
).withColumn("e_d", (F.col("c") + F.col("d")) * F.col("baseline_tokens") / F.col("N")).withColumn(
    "g2", 2 * (
        F.when(F.col("a") > 0, F.col("a") * F.log(F.col("a") / F.col("e_a"))).otherwise(0.0) +
        F.when(F.col("b") > 0, F.col("b") * F.log(F.col("b") / F.col("e_b"))).otherwise(0.0) +
        F.when(F.col("c") > 0, F.col("c") * F.log(F.col("c") / F.col("e_c"))).otherwise(0.0) +
        F.when(F.col("d") > 0, F.col("d") * F.log(F.col("d") / F.col("e_d"))).otherwise(0.0)
    )
).withColumn(
    "spike_rate", F.col("spike_tf") / F.col("spike_tokens")
).withColumn(
    "baseline_rate", F.col("baseline_tf") / F.col("baseline_tokens")
).withColumn(
    "frequency_ratio", (F.col("spike_rate") + F.lit(1e-12)) / (F.col("baseline_rate") + F.lit(1e-12))
).filter(F.col("spike_rate") > F.col("baseline_rate"))
key_rank_w = Window.partitionBy("spike_date").orderBy(F.desc("g2"), F.desc("frequency_ratio"), "term")
spike_key_terms = keyness.withColumn("rank", F.row_number().over(key_rank_w)).filter(F.col("rank") <= 10).select(
    F.col("spike_date").alias("publication_date"), "rank", "term", "spike_tf", "baseline_tf",
    F.round("frequency_ratio", 2).alias("frequency_ratio"), F.round("g2", 2).alias("g2")
).orderBy("publication_date", "rank")

# Backward-compatible names plus a compact explanation table.
spike_top_categories = spike_category_attribution
spike_top_terms = spike_key_terms

top_cat = spike_category_attribution.filter("rank = 1").select(
    "publication_date", F.col("category").alias("dominant_category"),
    F.round("dimension_z", 2).alias("category_z"), F.col("day_share_pct").alias("category_share_pct")
)
top_src = spike_source_attribution.filter("rank = 1").select(
    "publication_date", F.col("newspaper_source").alias("dominant_source"),
    F.round("dimension_z", 2).alias("source_z"), F.col("day_share_pct").alias("source_share_pct")
)
term_summary = spike_key_terms.groupBy("publication_date").agg(
    F.concat_ws(", ", F.sort_array(F.collect_list(F.struct("rank", "term"))).getField("term")).alias("key_terms")
)
spike_explanation = spikes.select(
    "publication_date", "daily_articles", F.round("z_score", 2).alias("overall_z")
).join(top_cat, "publication_date", "left").join(top_src, "publication_date", "left").join(
    source_agreement, "publication_date", "left"
).join(term_summary, "publication_date", "left").orderBy(F.desc("overall_z"))

display(spike_explanation.toPandas())
display(spike_category_attribution.toPandas())
display(spike_source_attribution.toPandas())
display(spike_key_terms.toPandas())


# Part G — Export tables and generate report-ready figures

In [ ]:
tables = {
    "category_distribution": category_summary.toPandas(),
    "source_distribution": source_summary.toPandas(),
    "yearly_counts": yearly.toPandas(),
    "monthly_counts": monthly.toPandas(),
    "seasonal_counts": seasonal.toPandas(),
    "weekday_weekend_counts": weekpart.toPandas(),
    "year_category_share": year_category_share.toPandas(),
    "category_yearly_growth": category_growth.toPandas(),
    "monthly_category_rolling": monthly_category_roll.toPandas(),
    "source_category_share": source_category_share.toPandas(),
    "publisher_summary": publisher_summary.toPandas(),
    "word_count_quantiles": quantile_table,
    "top_500_terms": word_freq.limit(500).toPandas(),
    "top_terms_by_category": top_terms_by_category.toPandas(),
    "tfidf_keywords_by_category": tfidf_keywords_pd,
    "keyword_evolution": keyword_evolution.toPandas(),
    "lda_model_comparison": lda_eval_pd,
    "lda_topic_terms": topic_terms_pd,
    "topic_prevalence_by_category": topic_by_category.toPandas(),
    "topic_prevalence_by_year": topic_by_year.toPandas(),
    "publication_spikes": spikes.toPandas(),
    "spike_top_categories": spike_top_categories.toPandas(),
    "spike_source_attribution": spike_source_attribution.toPandas(),
    "spike_source_agreement": source_agreement.toPandas(),
    "spike_top_terms": spike_top_terms.toPandas(),
    "spike_explanation": spike_explanation.toPandas(),
    "missing_values_before_cleaning": missing_before,
    "date_audit": date_audit_pd,
    "clean_validation": validation_pd,
}
for name, frame in tables.items():
    frame.to_csv(table_dir / f"{name}.csv", index=False, encoding="utf-8-sig")
print("CSV tables exported:", len(tables))

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
COLORS = ["#1565C0", "#00897B", "#F9A825", "#6A1B9A", "#D84315", "#2E7D32", "#455A64", "#C2185B"]

def savefig(name):
    plt.tight_layout()
    plt.savefig(fig_dir / name, dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close()

cat_pd = tables["category_distribution"].sort_values("count")
plt.figure(figsize=(10, 5.5)); plt.barh(cat_pd["category"], cat_pd["count"], color=COLORS[0])
plt.xlabel("Articles"); plt.title("Category distribution"); savefig("01_category_distribution.png")

src_pd = tables["source_distribution"].sort_values("count")
plt.figure(figsize=(10, 5)); plt.barh(src_pd["newspaper_source"], src_pd["count"], color=COLORS[1])
plt.xlabel("Articles"); plt.title("Newspaper distribution"); savefig("02_source_distribution.png")

yearly_pd = tables["yearly_counts"].sort_values("year")
plt.figure(figsize=(10, 5)); plt.plot(yearly_pd["year"], yearly_pd["count"], marker="o", color=COLORS[0])
plt.xlabel("Year"); plt.ylabel("Articles"); plt.title("Yearly publication volume"); savefig("03_yearly_volume.png")

yc_pd = year_category.toPandas()
plt.figure(figsize=(12, 7))
for color, (category_name, group) in zip(COLORS, yc_pd.groupby("category")):
    group = group.sort_values("year"); plt.plot(group["year"], group["count"], marker="o", label=category_name, color=color)
plt.xlabel("Year"); plt.ylabel("Articles"); plt.title("Category publication trends")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); savefig("04_category_trends.png")

source_pivot = tables["source_category_share"].pivot(
    index="newspaper_source", columns="category", values="source_category_pct"
).fillna(0)
plt.figure(figsize=(12, 5.5)); image = plt.imshow(source_pivot, cmap="Blues", aspect="auto")
plt.xticks(range(len(source_pivot.columns)), source_pivot.columns, rotation=35, ha="right")
plt.yticks(range(len(source_pivot.index)), source_pivot.index)
for y in range(len(source_pivot.index)):
    for x in range(len(source_pivot.columns)):
        value = source_pivot.iloc[y, x]
        plt.text(x, y, f"{value:.1f}", ha="center", va="center", fontsize=7,
                 color="white" if value > source_pivot.to_numpy().max() * 0.55 else "#222222")
plt.colorbar(image, label="Share (%)"); plt.title("Category share within each newspaper")
savefig("05_source_category_heatmap.png")

month_avg_pd = monthly.groupBy("month").agg(F.avg("count").alias("avg_articles")).orderBy("month").toPandas()
plt.figure(figsize=(10, 5)); plt.plot(month_avg_pd["month"], month_avg_pd["avg_articles"], marker="o", color=COLORS[2])
plt.xticks(range(1, 13)); plt.xlabel("Month"); plt.ylabel("Average articles"); plt.title("Average monthly seasonality")
savefig("06_monthly_seasonality.png")

kw_pd = tables["keyword_evolution"]
keyword_labels = {
    "বাংলাদেশ": "Bangladesh", "সরকার": "Government", "নির্বাচন": "Election", "শিক্ষা": "Education",
    "অর্থনীতি": "Economy", "খেলা": "Sports", "প্রযুক্তি": "Technology", "করোনা": "Coronavirus"
}
plt.figure(figsize=(12, 7))
for color, (term, group) in zip(COLORS, kw_pd.groupby("term")):
    group = group.sort_values("year"); plt.plot(group["year"], group["per_100k_tokens"], marker="o", label=keyword_labels.get(term, term), color=color)
plt.xlabel("Year"); plt.ylabel("Occurrences per 100,000 tokens"); plt.title("Selected keyword evolution")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); savefig("07_keyword_evolution.png")

plt.figure(figsize=(8, 5)); plt.plot(lda_eval_pd["k"], lda_eval_pd["log_perplexity"], marker="o", color=COLORS[3])
plt.xlabel("Topics (k)"); plt.ylabel("Log perplexity"); plt.title("LDA topic-count comparison")
savefig("08_lda_perplexity.png")

topic_year_pd = tables["topic_prevalence_by_year"]
plt.figure(figsize=(12, 7))
for topic, group in topic_year_pd.groupby("dominant_topic"):
    group = group.sort_values("year"); plt.plot(group["year"], group["year_topic_pct"], marker="o", label=f"Topic {topic}")
plt.xlabel("Year"); plt.ylabel("NLP sample share (%)"); plt.title("Topic prevalence over time")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); savefig("09_topic_prevalence.png")

daily_pd = daily_scored.select("publication_date", "daily_articles", "rolling_mean_28", "is_spike").orderBy("publication_date").toPandas()
spike_pd = daily_pd[daily_pd["is_spike"] == 1]
plt.figure(figsize=(14, 6)); plt.plot(daily_pd["publication_date"], daily_pd["daily_articles"], lw=.7, label="Daily articles")
plt.plot(daily_pd["publication_date"], daily_pd["rolling_mean_28"], lw=1.2, label="28-day historical mean")
plt.scatter(spike_pd["publication_date"], spike_pd["daily_articles"], s=20, color=COLORS[4], label="Candidate spikes")
plt.xlabel("Date"); plt.ylabel("Articles"); plt.title("Daily volume and publication spikes"); plt.legend()
savefig("10_publication_spikes.png")

# Part H — Validation, model export, and final package

In [ ]:
category_sum = category_summary.agg(F.sum("count")).first()[0]
source_sum = source_summary.agg(F.sum("count")).first()[0]
valid_year_rows = feature_df.filter("year is not null").count()
year_category_sum = year_category.agg(F.sum("count")).first()[0]
category_pct_sum = float(category_summary.agg(F.sum("percentage")).first()[0])

checks = {
    "category_counts_equal_clean_rows": category_sum == clean_count,
    "source_counts_equal_clean_rows": source_sum == clean_count,
    "year_category_counts_equal_dated_rows": year_category_sum == valid_year_rows,
    "category_percentages_approximately_100": abs(category_pct_sum - 100.0) <= 0.1,
    "eight_canonical_categories": category_summary.count() == 8,
    "nonempty_nlp_sample": nlp_count > 0,
    "lda_model_selected": BEST_K in LDA_K_VALUES,
}
assert all(checks.values()), checks

model_status = {}
for name, model, destination in [
    ("tfidf_countvectorizer", cv_model, model_dir / "tfidf_countvectorizer"),
    ("idf_model", idf_model, model_dir / "idf_model"),
    ("lda_countvectorizer", lda_cv_model, model_dir / "lda_countvectorizer"),
    (f"lda_model_k{BEST_K}", lda_model, model_dir / f"lda_model_k{BEST_K}"),
]:
    try:
        model.write().overwrite().save(str(destination)); model_status[name] = "saved"
    except Exception as exc:
        model_status[name] = f"not saved in this runtime: {type(exc).__name__}"

summary_metrics = {
    "raw_rows": int(raw_count), "clean_rows": int(clean_count),
    "rows_removed": int(raw_count - clean_count), "exact_duplicate_extra_rows": int(duplicate_extra_rows),
    "categories": int(category_summary.count()), "newspaper_sources": int(source_summary.count()),
    "min_date": str(min_date), "max_date": str(max_date),
    "nlp_sample_rows": int(nlp_count), "lda_training_rows": int(lda_train_count),
    "selected_lda_k": int(BEST_K), "candidate_spike_dates": int(spikes.count()),
    "tfidf_vocabulary_size": len(vocabulary), "lda_vocabulary_size": len(lda_vocab),
    "parquet_status": parquet_status, "model_status": model_status,
    "validation_checks": checks,
}
with (project_dir / "project_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(summary_metrics, handle, ensure_ascii=False, indent=2)

zip_path = project_dir.parent / "Bangla_News_Project_Results.zip"
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip", project_dir)
display(pd.DataFrame(summary_metrics.items(), columns=["metric", "value"]))
print("All validation checks passed.")
print("Figures:", len(list(fig_dir.glob("*.png"))))
print("Tables:", len(list(table_dir.glob("*.csv"))))
print("Results ZIP:", zip_path)

## Interpretation guide

- Category and publisher totals describe archive coverage, not public demand.
- Year-to-year changes can reflect collection coverage as well as editorial activity.
- TF–IDF and LDA results are estimated from the reported category-balanced sample.
- LDA topic IDs require human labels based on the exported top-word table.
- Spike dates identify unusual publication volume relative to the preceding 28 days; external
  evidence is required before linking a spike to a specific real-world event.
- Enhanced spike attribution scores each category and newspaper source against its own preceding 28-day baseline.
- Source agreement helps distinguish cross-publisher anomalies from spikes concentrated in one outlet.
- Spike keywords use full-corpus daily tokens and Dunning G² keyness against the same preceding 28-day window; they are explanatory signals, not causal proof.


# Part I - Event-calendar validation and topic coherence

Part F now explains *what* drove each spike. This part checks whether the detected spikes
correspond to anything real, and adds a human-interpretability check on the LDA topic count.

Both use only the Potrika corpus - no second dataset is required anywhere in this notebook.

## I.1 - Event-calendar validation

A **template** ground-truth calendar: a few well-documented national events plus recurring
national observances. This is a starting point, not an authoritative list - verify these
dates and add 20-30 more events relevant to your analysis before reporting the numbers as a
finding.

Because the calendar is incomplete, **precision here is a lower bound**: a flagged spike with
no matching calendar entry may still be a real but uncatalogued event, not a false positive.

In [ ]:
EVENT_MATCH_TOLERANCE_DAYS = 2  # match an event to any spike within +/- this many days

event_calendar_pd = pd.DataFrame(
    [
        {"event_date": "2016-07-01", "event_name": "Holey Artisan Bakery attack, Dhaka"},
        {"event_date": "2018-12-30", "event_name": "11th Bangladesh national election"},
        {"event_date": "2020-02-01", "event_name": "Dhaka city corporation elections"},
        {"event_date": "2020-03-08", "event_name": "First COVID-19 cases confirmed in Bangladesh"},
        {"event_date": "2020-03-26", "event_name": "Nationwide COVID-19 general holiday begins"},
    ]
    + [{"event_date": str(y) + "-02-21", "event_name": "Language Martyrs Day"} for y in range(2015, 2021)]
    + [{"event_date": str(y) + "-03-26", "event_name": "Independence Day"} for y in range(2015, 2021)]
    + [{"event_date": str(y) + "-12-16", "event_name": "Victory Day"} for y in range(2014, 2021)]
).drop_duplicates(subset=["event_date"]).reset_index(drop=True)

event_calendar_pd["event_date"] = pd.to_datetime(event_calendar_pd["event_date"]).dt.date
# Keep only events inside the corpus date range, otherwise recall is punished for dates the
# corpus could never have covered.
event_calendar_pd = event_calendar_pd[
    (event_calendar_pd["event_date"] >= min_date) & (event_calendar_pd["event_date"] <= max_date)
].reset_index(drop=True)
print("Events inside corpus range:", len(event_calendar_pd))
event_calendar_sdf = spark.createDataFrame(event_calendar_pd)


def evaluate_detector(flag_df, flag_col):
    """Match labeled events against flagged days and return per-event rows + summary metrics."""
    matched = event_calendar_sdf.alias("e").join(
        flag_df.select("publication_date", flag_col).alias("d"),
        F.abs(F.datediff(F.col("d.publication_date"), F.col("e.event_date"))) <= EVENT_MATCH_TOLERANCE_DAYS,
        "left",
    ).groupBy("event_date", "event_name").agg(
        F.coalesce(F.max(flag_col), F.lit(0)).alias("matched_spike")
    ).orderBy("event_date").toPandas()

    flagged_days = flag_df.filter(F.col(flag_col) == 1).count()
    tp = int(matched["matched_spike"].sum())
    fn = int((matched["matched_spike"] == 0).sum())
    fp = max(flagged_days - tp, 0)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    metrics = {
        "flagged_days": flagged_days, "events_matched": tp, "events_missed": fn,
        "precision_lower_bound": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3),
    }
    return matched, metrics


# Detector 1: the Part F rolling 28-day z-score
event_match_summary, rolling_metrics = evaluate_detector(daily_scored, "is_spike")

# Detector 2: naive whole-period fixed threshold, no rolling window (baseline for comparison)
wp = daily.agg(F.avg("daily_articles").alias("mu"), F.stddev_samp("daily_articles").alias("sigma")).first()
naive_threshold = wp["mu"] + 2 * wp["sigma"]
naive_scored = daily.withColumn(
    "naive_is_spike", (F.col("daily_articles") >= F.lit(naive_threshold)).cast("int")
)
naive_event_summary, naive_metrics = evaluate_detector(naive_scored, "naive_is_spike")

detector_comparison = pd.DataFrame([
    {"detector": "rolling_28day_zscore_ge_3", **rolling_metrics},
    {"detector": "naive_fixed_threshold_mean_plus_2sd", **naive_metrics},
])

display(event_match_summary)
display(detector_comparison)

## I.2 - LDA topic coherence (c_v)

Log-perplexity measures held-out predictive likelihood; it does not reliably track whether a
human would judge the topics interpretable. Adding c_v coherence gives a second axis for
reporting k. If the two disagree, report both rather than silently picking one.

In [ ]:
try:
    from gensim.corpora import Dictionary
    from gensim.models import CoherenceModel
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gensim"])
    from gensim.corpora import Dictionary
    from gensim.models import CoherenceModel

COHERENCE_SAMPLE_DOCS = int(os.environ.get("COHERENCE_SAMPLE_DOCS", "20000"))
coherence_docs_pd = tokenized_df.select("tokens_filtered").limit(COHERENCE_SAMPLE_DOCS).toPandas()
coherence_texts = [list(t) for t in coherence_docs_pd["tokens_filtered"] if t is not None and len(t) > 0]
coherence_dictionary = Dictionary(coherence_texts)

coherence_rows = []
for k, model in lda_models.items():
    topic_word_lists = [
        [lda_cv_model.vocabulary[int(idx)] for idx in row["termIndices"]]
        for row in model.describeTopics(maxTermsPerTopic=10).collect()
    ]
    score = CoherenceModel(
        topics=topic_word_lists, texts=coherence_texts,
        dictionary=coherence_dictionary, coherence="c_v"
    ).get_coherence()
    coherence_rows.append({"k": k, "coherence_c_v": round(float(score), 4)})
    print("k=" + str(k) + "  c_v coherence = " + format(score, ".4f"))

lda_eval_with_coherence = lda_eval_pd.merge(
    pd.DataFrame(coherence_rows).sort_values("k"), on="k"
)
BEST_K_COHERENCE = int(lda_eval_with_coherence.sort_values("coherence_c_v", ascending=False).iloc[0]["k"])
print("Lowest-perplexity k:", BEST_K, " | Highest-coherence k:", BEST_K_COHERENCE)
display(lda_eval_with_coherence)

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(lda_eval_with_coherence["k"], lda_eval_with_coherence["log_perplexity"],
         marker="o", color="#1565C0")
ax1.set_xlabel("Topics (k)")
ax1.set_ylabel("Log perplexity (lower = better)", color="#1565C0")
ax2 = ax1.twinx()
ax2.plot(lda_eval_with_coherence["k"], lda_eval_with_coherence["coherence_c_v"],
         marker="s", color="#D84315")
ax2.set_ylabel("c_v coherence (higher = better)", color="#D84315")
plt.title("LDA model selection: perplexity vs coherence")
fig.tight_layout()
plt.savefig(fig_dir / "11_lda_perplexity_vs_coherence.png", dpi=220, bbox_inches="tight", facecolor="white")
plt.show(); plt.close()

# Part K - Which newspaper leads? (cross-source lead-lag analysis)

**The question in one line:** when a story starts getting covered, which newspaper covers it
first, and which ones follow?

**How it works.** Part F already built, for every newspaper, a daily series of how unusual
that day's output was for *that* newspaper (`dimension_z` - its own 28-day rolling z-score).
Using the z-score rather than raw counts matters: it removes the huge volume differences
between outlets, so a small paper's busy day counts as much as a large paper's busy day.

For each ordered pair of newspapers (A, B) we correlate A's series with B's series shifted
forward by L days, for L in -MAX_LAG..+MAX_LAG. The L with the strongest correlation is the
pair's lead-lag: **L > 0 means B's coverage follows A's, so A leads B.**

A `lead_score` then counts, for each newspaper, how many other newspapers it leads minus how
many it follows.

**Read this with care.** This measures temporal precedence in publication volume, not
influence: two outlets can both react to the same press conference. Daily granularity is
coarse, so a true few-hour lead is invisible here. Treat significant lags as evidence of
coverage-cycle structure, not proof of agenda-setting.

In [ ]:
from scipy.stats import pearsonr, spearmanr

MAX_LAG_DAYS = int(os.environ.get("MAX_LAG_DAYS", "3"))
MIN_OVERLAP_DAYS = int(os.environ.get("MIN_OVERLAP_DAYS", "180"))

# source_daily_scored comes from Part F: one row per (date, newspaper) with that newspaper's
# own 28-day rolling z-score. Restrict to rows where a baseline actually existed.
source_series_pd = source_daily_scored.filter(
    F.col("dimension_z").isNotNull()
).select("publication_date", "newspaper_source", "dimension_z").toPandas()

source_matrix = source_series_pd.pivot(
    index="publication_date", columns="newspaper_source", values="dimension_z"
).sort_index()

# Only keep sources with enough scored days to make a correlation meaningful.
usable_sources = [c for c in source_matrix.columns if source_matrix[c].notna().sum() >= MIN_OVERLAP_DAYS]
source_matrix = source_matrix[usable_sources]
print("Sources in lead-lag analysis:", len(usable_sources), "|", ", ".join(map(str, usable_sources)))
print("Scored days available:", len(source_matrix))


def best_lag_for_pair(series_a, series_b, max_lag=MAX_LAG_DAYS, min_overlap=MIN_OVERLAP_DAYS):
    """Correlate A_t against B_{t+lag}. Positive best_lag => B follows A => A leads B.
    Returns the lag with the largest absolute correlation, plus its p-value and the lag-0
    correlation for comparison."""
    rows = []
    for lag in range(-max_lag, max_lag + 1):
        shifted_b = series_b.shift(-lag)
        pair = pd.concat([series_a, shifted_b], axis=1).dropna()
        if len(pair) < min_overlap:
            continue
        r, p = pearsonr(pair.iloc[:, 0], pair.iloc[:, 1])
        rows.append({"lag": lag, "corr": r, "p_value": p, "n": len(pair)})
    if not rows:
        return None
    frame = pd.DataFrame(rows)
    best = frame.loc[frame["corr"].abs().idxmax()]
    lag0 = frame[frame["lag"] == 0]
    return {
        "best_lag_days": int(best["lag"]),
        "corr_at_best_lag": round(float(best["corr"]), 4),
        "p_value_at_best_lag": float(best["p_value"]),
        "n_days": int(best["n"]),
        "corr_at_lag_0": round(float(lag0["corr"].iloc[0]), 4) if len(lag0) else None,
    }


lead_lag_rows = []
for i, a in enumerate(usable_sources):
    for b in usable_sources[i + 1:]:
        result = best_lag_for_pair(source_matrix[a], source_matrix[b])
        if result is None:
            continue
        if result["best_lag_days"] > 0:
            leader, follower = a, b
        elif result["best_lag_days"] < 0:
            leader, follower = b, a
        else:
            leader, follower = None, None
        lead_lag_rows.append({
            "source_a": a, "source_b": b, **result,
            "leader": leader, "follower": follower,
            "significant_at_0p01": bool(result["p_value_at_best_lag"] < 0.01),
        })

lead_lag_pairs = pd.DataFrame(lead_lag_rows).sort_values(
    "corr_at_best_lag", ascending=False
).reset_index(drop=True)
display(lead_lag_pairs)

# Lead score: only count pairs whose best-lag correlation is significant and non-zero-lag.
significant_pairs = lead_lag_pairs[
    lead_lag_pairs["significant_at_0p01"] & lead_lag_pairs["leader"].notna()
]
lead_counts = significant_pairs["leader"].value_counts()
follow_counts = significant_pairs["follower"].value_counts()
lead_score_pd = pd.DataFrame({
    "newspaper_source": usable_sources,
}).assign(
    leads_n=lambda d: d["newspaper_source"].map(lead_counts).fillna(0).astype(int),
    follows_n=lambda d: d["newspaper_source"].map(follow_counts).fillna(0).astype(int),
).assign(
    lead_score=lambda d: d["leads_n"] - d["follows_n"]
).sort_values("lead_score", ascending=False).reset_index(drop=True)
display(lead_score_pd)
print("Significant non-zero-lag pairs:", len(significant_pairs), "of", len(lead_lag_pairs))

In [ ]:
# Lead-lag heatmap: cell (row=leader, col=follower) shows the correlation at the pair's best
# lag when that pair is significant and the row leads the column; blank otherwise.
lead_matrix = pd.DataFrame(0.0, index=usable_sources, columns=usable_sources)
for _, row in significant_pairs.iterrows():
    lead_matrix.loc[row["leader"], row["follower"]] = row["corr_at_best_lag"]

plt.figure(figsize=(1.4 * len(usable_sources) + 4, 1.1 * len(usable_sources) + 3))
image = plt.imshow(lead_matrix, cmap="Oranges", aspect="auto", vmin=0)
plt.xticks(range(len(usable_sources)), usable_sources, rotation=35, ha="right")
plt.yticks(range(len(usable_sources)), usable_sources)
plt.xlabel("Follower"); plt.ylabel("Leader")
for y in range(len(usable_sources)):
    for x in range(len(usable_sources)):
        value = lead_matrix.iloc[y, x]
        if value > 0:
            plt.text(x, y, format(value, ".2f"), ha="center", va="center", fontsize=8,
                     color="white" if value > lead_matrix.to_numpy().max() * 0.6 else "#222222")
plt.colorbar(image, label="Correlation at best lag")
plt.title("Who leads whom: cross-newspaper lead-lag (significant pairs only)")
plt.tight_layout()
plt.savefig(fig_dir / "12_lead_lag_matrix.png", dpi=220, bbox_inches="tight", facecolor="white")
plt.show(); plt.close()

plt.figure(figsize=(9, 5))
plt.barh(lead_score_pd["newspaper_source"], lead_score_pd["lead_score"], color="#00897B")
plt.axvline(0, color="#444444", lw=0.8)
plt.xlabel("Lead score  (newspapers led  -  newspapers followed)")
plt.title("Which newspapers lead the coverage cycle")
plt.tight_layout()
plt.savefig(fig_dir / "13_lead_score.png", dpi=220, bbox_inches="tight", facecolor="white")
plt.show(); plt.close()

In [ ]:
# Per-category lead-lag: agenda-setting may differ by beat (e.g. Politics vs Sports).
LEAD_LAG_CATEGORIES = [
    r["category"] for r in category_summary.orderBy(F.desc("count")).limit(4).collect()
]
print("Categories analysed:", ", ".join(LEAD_LAG_CATEGORIES))

category_source_daily = feature_df.filter("publication_date is not null").filter(
    F.col("category").isin(LEAD_LAG_CATEGORIES)
).groupBy("publication_date", "category", "newspaper_source").count().withColumnRenamed("count", "n_articles")

cat_src_w = Window.partitionBy("category", "newspaper_source").orderBy("publication_date").rowsBetween(-28, -1)
category_source_scored = category_source_daily.withColumn(
    "baseline_n", F.count("n_articles").over(cat_src_w)
).withColumn("baseline_mean", F.avg("n_articles").over(cat_src_w)
).withColumn("baseline_std", F.stddev_samp("n_articles").over(cat_src_w)
).withColumn(
    "cat_src_z",
    F.when((F.col("baseline_n") >= 14) & (F.col("baseline_std") > 0),
           (F.col("n_articles") - F.col("baseline_mean")) / F.col("baseline_std"))
)

category_series_pd = category_source_scored.filter(
    F.col("cat_src_z").isNotNull()
).select("publication_date", "category", "newspaper_source", "cat_src_z").toPandas()

category_lead_rows = []
for cat in LEAD_LAG_CATEGORIES:
    subset = category_series_pd[category_series_pd["category"] == cat]
    matrix = subset.pivot(index="publication_date", columns="newspaper_source", values="cat_src_z").sort_index()
    cols = [c for c in matrix.columns if matrix[c].notna().sum() >= MIN_OVERLAP_DAYS]
    matrix = matrix[cols]
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            result = best_lag_for_pair(matrix[a], matrix[b])
            if result is None or result["best_lag_days"] == 0:
                continue
            leader, follower = (a, b) if result["best_lag_days"] > 0 else (b, a)
            category_lead_rows.append({
                "category": cat, "leader": leader, "follower": follower, **result,
                "significant_at_0p01": bool(result["p_value_at_best_lag"] < 0.01),
            })

category_lead_lag = pd.DataFrame(category_lead_rows)
if len(category_lead_lag):
    category_lead_lag = category_lead_lag.sort_values(
        ["category", "corr_at_best_lag"], ascending=[True, False]
    ).reset_index(drop=True)
    category_leaders = category_lead_lag[category_lead_lag["significant_at_0p01"]].groupby(
        ["category", "leader"]
    ).size().reset_index(name="times_leading").sort_values(
        ["category", "times_leading"], ascending=[True, False]
    ).reset_index(drop=True)
    display(category_lead_lag.head(40))
    display(category_leaders)
else:
    category_leaders = pd.DataFrame(columns=["category", "leader", "times_leading"])
    print("No qualifying category-level pairs. Try lowering MIN_OVERLAP_DAYS.")

# Part L - Do headlines match their own articles? (headline-body divergence)

**The question in one line:** when a Bangla newspaper writes a headline, how much of it is
actually supported by the article underneath it?

**How it works.** For every article we tokenise the headline and the body separately with the
same tokenizer and Bangla stopword list used elsewhere, then compute:

```
divergence = (headline content-words NOT appearing in the body) / (headline content-words)
```

0.0 means every headline word is grounded in the article; 1.0 means none are. This needs **no
labeled data**, which is the point: clickbait/sensationalism work in Bangla normally requires
hand annotation, and this is an unsupervised proxy that scales to the whole corpus.

We then aggregate by newspaper, by category, and by newspaper-year (to check the ranking is
stable over time rather than an artifact of one period), and export the highest- and
lowest-divergence headlines so the measure can be checked by hand.

**Read this with care.** High divergence is not proof of clickbait. A headline can legitimately
paraphrase, use a synonym, or name something the body refers to differently - Bangla
morphology means the same lemma can surface in inflected forms this exact-match measure will
miss. So this measures *lexical* grounding, and the manual-check sample below is what tells
you whether high divergence really corresponds to sensational headlines in your corpus.

In [ ]:
HEADLINE_MIN_TOKENS = int(os.environ.get("HEADLINE_MIN_TOKENS", "3"))

headline_tokenizer = RegexTokenizer(
    inputCol="headline_clean", outputCol="headline_tokens_raw",
    pattern=r"[^\u0980-\u09FFA-Za-z0-9]+", gaps=True, minTokenLength=2, toLowercase=True
)
headline_remover = StopWordsRemover(
    inputCol="headline_tokens_raw", outputCol="headline_tokens",
    stopWords=BANGLA_STOPWORDS, caseSensitive=False
)

# Body tokens come from the SAME tokenizer/stopword list used in Part C, so the two sides are
# directly comparable. This runs on the full cleaned corpus, not the NLP sample.
divergence_base = feature_df.filter(
    F.col("headline_clean").isNotNull() & (F.length("headline_clean") > 0)
).select(
    "publication_date", "year", "category", "newspaper_source",
    "headline", "headline_clean", "news_article_clean", "word_count"
)

divergence_tokenized = headline_remover.transform(
    headline_tokenizer.transform(
        remover.transform(tokenizer.transform(divergence_base))
    )
)

divergence_df = divergence_tokenized.withColumn(
    "headline_terms", F.array_distinct("headline_tokens")
).withColumn(
    "body_terms", F.array_distinct("tokens_filtered")
).withColumn(
    "n_headline_terms", F.size("headline_terms")
).filter(
    F.col("n_headline_terms") >= HEADLINE_MIN_TOKENS
).withColumn(
    "n_unmatched", F.size(F.array_except("headline_terms", "body_terms"))
).withColumn(
    "headline_body_divergence", F.round(F.col("n_unmatched") / F.col("n_headline_terms"), 4)
).persist(StorageLevel.DISK_ONLY)

divergence_rows = divergence_df.count()
print("Articles scored for headline-body divergence:", format(divergence_rows, ","))
display(divergence_df.select(
    "newspaper_source", "category", "headline", "n_headline_terms",
    "n_unmatched", "headline_body_divergence"
).limit(10).toPandas())

In [ ]:
# Aggregate by newspaper, by category, and by newspaper-year (stability check).
divergence_by_source = divergence_df.groupBy("newspaper_source").agg(
    F.count("*").alias("n_articles"),
    F.round(F.avg("headline_body_divergence"), 4).alias("mean_divergence"),
    F.round(F.expr("percentile_approx(headline_body_divergence, 0.5)"), 4).alias("median_divergence"),
    F.round(F.stddev_samp("headline_body_divergence"), 4).alias("sd_divergence"),
    F.round(F.avg("n_headline_terms"), 2).alias("avg_headline_terms"),
    F.round(F.avg("word_count"), 1).alias("avg_body_words"),
).orderBy(F.desc("mean_divergence"))

divergence_by_category = divergence_df.groupBy("category").agg(
    F.count("*").alias("n_articles"),
    F.round(F.avg("headline_body_divergence"), 4).alias("mean_divergence"),
).orderBy(F.desc("mean_divergence"))

divergence_by_source_year = divergence_df.filter("year is not null").groupBy(
    "newspaper_source", "year"
).agg(
    F.count("*").alias("n_articles"),
    F.round(F.avg("headline_body_divergence"), 4).alias("mean_divergence"),
).orderBy("newspaper_source", "year")

divergence_by_source_pd = divergence_by_source.toPandas()
divergence_by_category_pd = divergence_by_category.toPandas()
divergence_by_source_year_pd = divergence_by_source_year.toPandas()

display(divergence_by_source_pd)
display(divergence_by_category_pd)

# Stability: is the per-newspaper ranking consistent year to year? Report the average pairwise
# Spearman correlation between consecutive years' rankings. High value => a stable trait of the
# outlet, not a one-off period effect.
rank_pivot = divergence_by_source_year_pd.pivot(
    index="newspaper_source", columns="year", values="mean_divergence"
)
years_available = sorted([c for c in rank_pivot.columns if rank_pivot[c].notna().sum() >= 3])
stability_rows = []
for y1, y2 in zip(years_available, years_available[1:]):
    pair = rank_pivot[[y1, y2]].dropna()
    if len(pair) >= 3:
        rho, p = spearmanr(pair[y1], pair[y2])
        stability_rows.append({"year_a": y1, "year_b": y2, "spearman_rho": round(float(rho), 3),
                               "p_value": round(float(p), 4), "n_sources": len(pair)})
divergence_rank_stability = pd.DataFrame(stability_rows)
display(divergence_rank_stability)
if len(divergence_rank_stability):
    print("Mean year-to-year rank correlation:",
          round(float(divergence_rank_stability["spearman_rho"].mean()), 3))

In [ ]:
# Manual-validation sample: the highest- and lowest-divergence headlines. Read these by hand -
# this is what tells you whether the measure is capturing sensationalism or just paraphrase.
VALIDATION_SAMPLE_N = int(os.environ.get("VALIDATION_SAMPLE_N", "50"))

divergence_ranked = divergence_df.filter(F.col("n_headline_terms") >= 4).select(
    "publication_date", "newspaper_source", "category", "headline",
    "n_headline_terms", "n_unmatched", "headline_body_divergence", "word_count"
)
high_divergence_sample = divergence_ranked.orderBy(
    F.desc("headline_body_divergence"), F.desc("n_headline_terms")
).limit(VALIDATION_SAMPLE_N).toPandas()
low_divergence_sample = divergence_ranked.orderBy(
    "headline_body_divergence", F.desc("n_headline_terms")
).limit(VALIDATION_SAMPLE_N).toPandas()
divergence_validation_sample = pd.concat([
    high_divergence_sample.assign(sample_group="high_divergence"),
    low_divergence_sample.assign(sample_group="low_divergence"),
], ignore_index=True)
display(divergence_validation_sample.head(20))

plt.figure(figsize=(10, 5))
plot_src = divergence_by_source_pd.sort_values("mean_divergence")
plt.barh(plot_src["newspaper_source"], plot_src["mean_divergence"], color="#6A1B9A")
plt.xlabel("Mean headline-body divergence  (0 = fully grounded, 1 = no overlap)")
plt.title("How well each newspaper's headlines match its own articles")
plt.tight_layout()
plt.savefig(fig_dir / "14_headline_divergence_by_source.png", dpi=220, bbox_inches="tight", facecolor="white")
plt.show(); plt.close()

plt.figure(figsize=(11, 6))
for color, (src, group) in zip(COLORS, divergence_by_source_year_pd.groupby("newspaper_source")):
    group = group.sort_values("year")
    plt.plot(group["year"], group["mean_divergence"], marker="o", label=src, color=color)
plt.xlabel("Year"); plt.ylabel("Mean divergence")
plt.title("Headline-body divergence over time, by newspaper")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(fig_dir / "15_headline_divergence_over_time.png", dpi=220, bbox_inches="tight", facecolor="white")
plt.show(); plt.close()

## L.1 - Does headline divergence change during spikes?

This links Part L back to Part F: on days the corpus spikes (breaking-news days), do headlines
become *less* grounded in their articles than usual? This is the one place the two new
contributions touch, and it is a genuinely new question - it asks whether editorial behaviour
shifts under breaking-news pressure.

In [ ]:
divergence_daily = divergence_df.groupBy("publication_date").agg(
    F.count("*").alias("n_articles"),
    F.avg("headline_body_divergence").alias("mean_divergence"),
)
divergence_spike_flagged = divergence_daily.join(
    daily_scored.select("publication_date", "is_spike"), "publication_date", "inner"
).filter(F.col("n_articles") >= 20)

spike_vs_normal = divergence_spike_flagged.groupBy("is_spike").agg(
    F.count("*").alias("n_days"),
    F.round(F.avg("mean_divergence"), 4).alias("mean_daily_divergence"),
    F.round(F.stddev_samp("mean_divergence"), 4).alias("sd_daily_divergence"),
).orderBy("is_spike").toPandas()
display(spike_vs_normal)

# Welch t-test: is the difference larger than day-to-day noise?
from scipy.stats import ttest_ind
flagged_pd = divergence_spike_flagged.select("is_spike", "mean_divergence").toPandas()
spike_values = flagged_pd.loc[flagged_pd["is_spike"] == 1, "mean_divergence"]
normal_values = flagged_pd.loc[flagged_pd["is_spike"] == 0, "mean_divergence"]

if len(spike_values) >= 5:
    t_stat, p_value = ttest_ind(spike_values, normal_values, equal_var=False)
    pooled_sd = np.sqrt((spike_values.var(ddof=1) + normal_values.var(ddof=1)) / 2)
    cohens_d = (spike_values.mean() - normal_values.mean()) / pooled_sd if pooled_sd > 0 else np.nan
    spike_divergence_test = pd.DataFrame([{
        "spike_days": len(spike_values), "normal_days": len(normal_values),
        "mean_divergence_spike": round(float(spike_values.mean()), 4),
        "mean_divergence_normal": round(float(normal_values.mean()), 4),
        "difference": round(float(spike_values.mean() - normal_values.mean()), 4),
        "welch_t": round(float(t_stat), 3), "p_value": round(float(p_value), 5),
        "cohens_d": round(float(cohens_d), 3),
    }])
    display(spike_divergence_test)
    print("Note: with few spike days this test is underpowered; report the effect size (cohens_d)")
    print("alongside p, and do not claim an effect from p alone.")
else:
    spike_divergence_test = pd.DataFrame()
    print("Too few spike days with >=20 scored articles to run a meaningful test.")

# Part M - Export the new tables

Adds every Part I / K / L table to the same `table_dir` used by Part G, so they land in the
final results ZIP built in Part H. If you have already run Part H, re-run its last cell to
refresh the archive.

In [ ]:
research_tables = {
    # Part I - validation
    "event_calendar_matches": event_match_summary,
    "detector_comparison_precision_recall": detector_comparison,
    "lda_perplexity_vs_coherence": lda_eval_with_coherence,
    # Part K - lead-lag
    "leadlag_pairs": lead_lag_pairs,
    "leadlag_lead_score": lead_score_pd,
    "leadlag_by_category": category_lead_lag,
    "leadlag_category_leaders": category_leaders,
    # Part L - headline divergence
    "headline_divergence_by_source": divergence_by_source_pd,
    "headline_divergence_by_category": divergence_by_category_pd,
    "headline_divergence_by_source_year": divergence_by_source_year_pd,
    "headline_divergence_rank_stability": divergence_rank_stability,
    "headline_divergence_validation_sample": divergence_validation_sample,
    "headline_divergence_spike_vs_normal": spike_vs_normal,
    "headline_divergence_spike_test": spike_divergence_test,
}
exported = 0
for name, frame in research_tables.items():
    if frame is None or len(frame) == 0:
        print("Skipped (empty):", name)
        continue
    frame.to_csv(table_dir / (name + ".csv"), index=False, encoding="utf-8-sig")
    exported += 1
print("New research tables exported:", exported, "of", len(research_tables))
print("Figures now in output folder:", len(list(fig_dir.glob("*.png"))))
print("Tables now in output folder:", len(list(table_dir.glob("*.csv"))))

# How to describe this work (contribution summary)

Use this framing when writing the paper. Every claim below is supported by a table or figure
this notebook produces, from a single corpus.

**Contribution 1 - Explainable spike detection (Parts F + I).**
Coverage spikes are detected against a rolling 28-day baseline, then *attributed*: which
category, which newspaper, and which terms were unusually elevated, each measured against its
own history rather than the corpus average. Spikes are then validated against a calendar of
known events and compared with a naive fixed-threshold detector.
*Tables:* `spike_explanation`, `event_calendar_matches`, `detector_comparison_precision_recall`.

**Contribution 2 - Cross-newspaper lead-lag structure (Part K).**
Using each newspaper's own normalised daily activity, we measure which outlets' coverage
systematically precedes which others', overall and per category.
*Tables:* `leadlag_pairs`, `leadlag_lead_score`, `leadlag_category_leaders`.
*Figures:* `12_lead_lag_matrix.png`, `13_lead_score.png`.

**Contribution 3 - Unsupervised headline-body divergence (Part L).**
A label-free measure of how far each headline's vocabulary is grounded in its own article,
aggregated per outlet and per category, checked for year-to-year stability, and tested for
whether headlines become less grounded on breaking-news days.
*Tables:* `headline_divergence_by_source`, `headline_divergence_rank_stability`,
`headline_divergence_spike_test`.
*Figures:* `14_headline_divergence_by_source.png`, `15_headline_divergence_over_time.png`.

**What to claim, and what not to.** The individual techniques here - rolling z-scores, Dunning
G2 keyness, lagged cross-correlation, set-overlap similarity - are all standard. The
contribution is their combination and their first application at this scale to Bangla news,
plus the empirical findings about Bangladeshi media behaviour that result. Frame it as an
empirical contribution, not a new algorithm, and a reviewer has nothing to push back on.

**Limitations to state explicitly in the paper.**
- Spike attribution is relative: a flagged category or source is unusual *for itself*, not the
  largest in the corpus.
- Event-calendar precision is a lower bound because the calendar is incomplete.
- Lead-lag measures temporal precedence in publication volume, not influence; daily
  granularity hides sub-day leads; both outlets may simply react to a common source.
- Headline divergence measures lexical grounding only; Bangla inflection and legitimate
  paraphrase both inflate it, which is why the manual-validation sample matters.
- All per-source results are conditioned on how completely each newspaper was scraped in
  Potrika; uneven coverage across outlets is a property of the corpus, not of the press.

# Reviewer-response robustness checks
Run the original notebook above first. These appended cells address the expanded LDA grid, BH-FDR lead--lag correction, and Bangla suffix-normalization sensitivity check requested during manuscript review.


In [ ]:
# 0. Shared setup: run AFTER the executed original notebook, in the SAME Colab runtime.
# If the runtime was restarted, run your original notebook again first.
import numpy as np
import pandas as pd
import math, json, unicodedata
from pathlib import Path
from itertools import combinations
from scipy.stats import pearsonr
from pyspark.sql import functions as F, Window
from pyspark.storagelevel import StorageLevel
from pyspark.ml.clustering import LDA
from gensim.models import CoherenceModel

needed = ['lda_train','lda_models','lda_cv_model','coherence_texts',
          'coherence_dictionary','SEED','LDA_MAX_ITER','source_matrix',
          'feature_df','calendar_df','divergence_df','table_dir', 'fig_dir']
missing=[x for x in needed if x not in globals()]
if missing:
    raise RuntimeError('Run original notebook first in this runtime. Missing: '+', '.join(missing))
review_dir=Path(table_dir).parent/'reviewer_checks'
review_dir.mkdir(parents=True, exist_ok=True)
print('Reviewer results directory:', review_dir)
print('Original LDA training rows:', lda_train.count())
print('Expected original NLP sample:', tokenized_df.count())
print('Spark runtime:', spark.version)


In [ ]:
# Reviewer check 1: reproducible 80/20 split, freshly train ALL candidate K values.
# IMPORTANT: Original K=5/8/10 models were fit on the full lda_train and thus
# cannot be scored as held-out on any subset of that same training population.
# This cell trains new comparable models on the SAME 80% training subset and
# evaluates every candidate on the SAME unseen 20% holdout.
from pyspark.storagelevel import StorageLevel
from pyspark.ml.clustering import LDA
from gensim.models import CoherenceModel
import pandas as pd, matplotlib.pyplot as plt

lda_k_extended=[5,8,10,15,20,25]
lda_train80,lda_test20=lda_train.randomSplit([0.8,0.2],seed=SEED+901)
lda_train80=lda_train80.persist(StorageLevel.DISK_ONLY)
lda_test20=lda_test20.persist(StorageLevel.DISK_ONLY)
n_train=lda_train80.count(); n_test=lda_test20.count()
print(f'Expanded LDA split: {n_train:,} train / {n_test:,} held out')
if n_train<1000 or n_test<1000:
    raise RuntimeError('Too few rows in the held-out LDA split.')
rows=[]
for k in lda_k_extended:
    print('Training fresh held-out-grid model K=',k)
    model=LDA(k=k,maxIter=LDA_MAX_ITER,featuresCol='lda_features',
              seed=SEED,optimizer='online').fit(lda_train80)
    topics=[[lda_cv_model.vocabulary[int(idx)] for idx in t['termIndices']]
            for t in model.describeTopics(maxTermsPerTopic=10).collect()]
    coherence=float(CoherenceModel(topics=topics,texts=coherence_texts,
                         dictionary=coherence_dictionary,coherence='c_v').get_coherence())
    rows.append({'k':k,'train_n':n_train,'test_n':n_test,
                 'heldout_log_perplexity':float(model.logPerplexity(lda_test20)),
                 'training_log_perplexity':float(model.logPerplexity(lda_train80)),
                 'coherence_cv':coherence})
    print(f'K={k}, held-out log perplexity={rows[-1]["heldout_log_perplexity"]:.4f}, '
          f'coherence={coherence:.4f}')
    del model
expanded_lda_pd=pd.DataFrame(rows).sort_values('k')
expanded_lda_pd.to_csv(review_dir/'expanded_lda_HELDOUT_grid.csv',index=False)
display(expanded_lda_pd)
fig,ax=plt.subplots(figsize=(6,3.5),dpi=180)
ax.plot(expanded_lda_pd.k,expanded_lda_pd.heldout_log_perplexity,marker='o',
        label='Held-out log perplexity')
ax.set_xlabel('Topics (K)'); ax.set_ylabel('Held-out log perplexity (lower better)')
ax.set_xticks(lda_k_extended)
fig.tight_layout()
fig.savefig(review_dir/'expanded_lda_HELDOUT_grid.pdf',bbox_inches='tight')
plt.close(fig)
# Do not declare a global optimum unless the new results justify it.


In [ ]:
# 2. Recompute all seven lags x all pairs x four largest categories, then BH-FDR.
# All tests (including lag 0) belong to ONE multiplicity family.
# Category series are zero-filled and start at each source/category's first observed date.
# The reference notebook omitted zero-count category days, so corrected results may differ.

def get_lag_tests(matrix, category_name, min_days=180, max_lag=3):
    tests=[]
    sources=[c for c in matrix.columns if matrix[c].notna().sum()>=min_days]
    for a,b in combinations(sorted(sources),2):
        for lag in range(-max_lag,max_lag+1):
            pair=pd.concat([matrix[a],matrix[b].shift(-lag)],axis=1).dropna()
            if len(pair)<min_days or pair.iloc[:,0].nunique()<2 or pair.iloc[:,1].nunique()<2:
                continue
            r,p=pearsonr(pair.iloc[:,0].astype(float),pair.iloc[:,1].astype(float))
            if np.isfinite(r) and np.isfinite(p):
                tests.append({'category':category_name,'source_a':a,'source_b':b,
                              'lag':int(lag),'n_overlap':len(pair),
                              'pearson_r':float(r),'p_raw':float(p)})
    return tests

# Restrict each source to the observable first--last publication period; do not
# treat every pre-onboarding/post-last-publication day as a genuine zero.
source_coverage_pd=(feature_df.filter(F.col('publication_date').isNotNull())
  .groupBy('newspaper_source')
  .agg(F.min('publication_date').alias('first_date'),
       F.max('publication_date').alias('last_date'),
       F.countDistinct('publication_date').alias('active_days'))
  .toPandas())
source_coverage_pd.to_csv(review_dir/'source_coverage_windows.csv',index=False)
source_matrix_corrected=source_matrix.copy()
source_matrix_corrected.index=pd.to_datetime(source_matrix_corrected.index)
for _,v in source_coverage_pd.iterrows():
    if v['newspaper_source'] in source_matrix_corrected:
        outside=(source_matrix_corrected.index<pd.Timestamp(v['first_date'])) | (source_matrix_corrected.index>pd.Timestamp(v['last_date']))
        source_matrix_corrected.loc[outside,v['newspaper_source']]=np.nan
all_tests = get_lag_tests(source_matrix_corrected, 'Overall', min_days=MIN_OVERLAP_DAYS, max_lag=MAX_LAG_DAYS)

# Ensure actual complete date rows by category/source, but do not count dates
# before a source/category is first seen as its historical publication activity.
cats=LEAD_LAG_CATEGORIES  # from base notebook: National / International / Sports / Economy
counted=(feature_df.filter(F.col('publication_date').isNotNull())
         .filter(F.col('category').isin(cats))
         .groupBy('publication_date','category','newspaper_source')
         .count().withColumnRenamed('count','n_articles'))
first_dates=(counted.groupBy('category','newspaper_source')
             .agg(F.min('publication_date').alias('first_observed_date'),
               F.max('publication_date').alias('last_observed_date')))
cat_grid=(calendar_df.crossJoin(first_dates)
          .filter((F.col('publication_date')>=F.col('first_observed_date')) &
                  (F.col('publication_date')<=F.col('last_observed_date')))
          .select('publication_date','category','newspaper_source'))
cat_full=cat_grid.join(counted,['publication_date','category','newspaper_source'],'left')
cat_full=cat_full.fillna({'n_articles':0})
w=(Window.partitionBy('category','newspaper_source')
   .orderBy('publication_date').rowsBetween(-28,-1))
cat_full=(cat_full.withColumn('n_history',F.count('n_articles').over(w))
          .withColumn('mean_28',F.avg('n_articles').over(w))
          .withColumn('sd_28',F.stddev_samp('n_articles').over(w)))
cat_full=cat_full.withColumn('z',F.when((F.col('n_history')>=14)&(F.col('sd_28')>0),
           (F.col('n_articles')-F.col('mean_28'))/F.col('sd_28')))
cat_pd=cat_full.filter(F.col('z').isNotNull()).select(
    'publication_date','category','newspaper_source','z').toPandas()
cat_pd['publication_date']=pd.to_datetime(cat_pd['publication_date'])

for cat in cats:
    x=cat_pd[cat_pd['category']==cat]
    if not len(x):
        continue
    mat=x.pivot(index='publication_date',columns='newspaper_source',values='z').sort_index()
    mat=mat.reindex(pd.date_range(mat.index.min(),mat.index.max(),freq='D'))
    all_tests.extend(get_lag_tests(mat,cat,min_days=MIN_OVERLAP_DAYS,max_lag=MAX_LAG_DAYS))

lag_fdr_pd=pd.DataFrame(all_tests)
if lag_fdr_pd.empty:
    raise RuntimeError('No eligible lag tests. Check calendar data and threshold.')

# BH, including correction for trying different lags before choosing the best.
# Pure NumPy implementation to avoid a new statsmodels dependency.
def bh_adjust(p):
    p=np.asarray(p,dtype=float)
    order=np.argsort(p,kind='stable')
    ranks=np.arange(1,len(p)+1)
    qsort=np.minimum.accumulate((p[order]*len(p)/ranks)[::-1])[::-1]
    q=np.empty_like(p)
    q[order]=np.minimum(1.0,qsort)
    return q

lag_fdr_pd['q_bh_all_tests']=bh_adjust(lag_fdr_pd['p_raw'])
lag_fdr_pd['bh_reject_0p05']=lag_fdr_pd['q_bh_all_tests']<0.05
lag_fdr_pd['positive_nonzero']=((lag_fdr_pd['lag']!=0)&(lag_fdr_pd['pearson_r']>0))
lag_fdr_pd.to_csv(review_dir/'leadlag_all_lags_bh.csv',index=False)

# Pair-level summary: best ABS correlation is selected only AFTER correcting every test.
idx=lag_fdr_pd.groupby(['category','source_a','source_b'])['pearson_r'].apply(
    lambda s: s.abs().idxmax()).to_numpy(dtype=int)
pair_summary=lag_fdr_pd.loc[idx].copy().sort_values(['category','source_a','source_b'])
pair_summary['bh_confirmed_positive_lead']=(
    pair_summary['positive_nonzero'] & pair_summary['bh_reject_0p05'])
pair_summary['candidate_leader']=np.where(
    (pair_summary['lag']>0)&(pair_summary['pearson_r']>0), pair_summary['source_a'],
    np.where((pair_summary['lag']<0)&(pair_summary['pearson_r']>0), pair_summary['source_b'],''))
pair_summary['candidate_follower']=np.where(
    (pair_summary['lag']>0)&(pair_summary['pearson_r']>0), pair_summary['source_b'],
    np.where((pair_summary['lag']<0)&(pair_summary['pearson_r']>0), pair_summary['source_a'],''))
pair_summary.to_csv(review_dir/'leadlag_pair_summary_bh.csv',index=False)

summary=(pair_summary.groupby('category').agg(
    n_pairs=('lag','size'), n_best_zero_lag=('lag',lambda v:int((v==0).sum())),
    n_positive_nonzero_selected=('positive_nonzero','sum'),
    n_bh_confirmed_positive_lead=('bh_confirmed_positive_lead','sum'))
    .reset_index())
summary.to_csv(review_dir/'leadlag_bh_summary.csv',index=False)
print('Total tests corrected together:',len(lag_fdr_pd))
print('Selected-pair BH-FDR summary (positive r and nonzero lag only):')
display(summary)
print('Note: Pearson p-values remain susceptible to temporal autocorrelation even after FDR.')


In [ ]:
# 3. Paired lexical-overlap vs conservative Bangla suffix-normalized overlap.
# No raw news text is written to the results folder.
import unicodedata
suffixes=sorted([
    'গুলোর','গুলিকে','গুলোতে','গুলির','গুলো','গুলা','গুলিকে',
    'গুলির','গুলিতে','গুলি','দেরকে','দের','টির','টিকে','টিতে',
    'টার','টাকে','গুলোয়','টায়','টি','টা','কে','তে','রা',
],key=len,reverse=True)

def bn_suffix_normalize(term):
    t=unicodedata.normalize('NFC',str(term)).strip()
    # Do not stem English words/numerals or punctuation.
    if not t or not all('\u0980'<=ch<='\u09ff' for ch in t):
        return t
    for suffix in suffixes:
        if t.endswith(suffix) and len(t)-len(suffix)>=3:
            return t[:-len(suffix)]
    return t

def lexical_divergence(hterms,bterms,normalize=False):
    if normalize:
        h={bn_suffix_normalize(t) for t in hterms}
        b={bn_suffix_normalize(t) for t in bterms}
    else:
        h=set(hterms)
        b=set(bterms)
    return (len(h-b)/len(h)) if h else np.nan

counts={r['category']:r['count'] for r in divergence_df.groupBy('category').count().collect()}
fractions={k:min(1.0,1500.0/v) for k,v in counts.items() if v}
robust_sample=(divergence_df.sampleBy('category',fractions,seed=SEED)
               .select('category','newspaper_source','headline_terms','body_terms',
                       'headline_body_divergence').toPandas())
if len(robust_sample)<1000:
    raise RuntimeError('Unusually small morphology sample; check divergence_df.')
robust_sample['exact_recomputed']=[lexical_divergence(h,b) for h,b in zip(
    robust_sample['headline_terms'],robust_sample['body_terms'])]
robust_sample['suffix_normalized']=[lexical_divergence(h,b,True) for h,b in zip(
    robust_sample['headline_terms'],robust_sample['body_terms'])]
robust_sample['drop_exact_minus_suffix']=(
    robust_sample['exact_recomputed']-robust_sample['suffix_normalized'])
check_diff=(robust_sample['exact_recomputed']-
            robust_sample['headline_body_divergence'].astype(float)).abs().max()
print('Maximum recomputed-vs-original exact-score gap:',check_diff)
assert check_diff<=0.00011, 'Tokenization mismatch: inspect source divergence implementation'

# Category-balanced sample intentionally differs from corpus-weighted means.
delta=robust_sample['drop_exact_minus_suffix'].to_numpy(dtype=float)
rng=np.random.default_rng(SEED)
boot=np.array([rng.choice(delta,len(delta),replace=True).mean() for _ in range(500)])
lo,hi=np.quantile(boot,[0.025,0.975])
morph_summary=pd.DataFrame([{
    'n_sample':len(robust_sample), 'mean_exact':robust_sample['exact_recomputed'].mean(),
    'mean_suffix_normalized':robust_sample['suffix_normalized'].mean(),
    'mean_drop':delta.mean(), 'median_drop':np.median(delta),
    'share_articles_reduced':np.mean(delta>0), 'bootstrap95_low':lo,
    'bootstrap95_high':hi,
    'warning':'Heuristic suffix stripping, not a validated Bangla stemmer; categories balanced',
}])
morph_by_source=robust_sample.groupby('newspaper_source').agg(
    n=('exact_recomputed','size'),mean_exact=('exact_recomputed','mean'),
    mean_suffix=('suffix_normalized','mean'), mean_drop=('drop_exact_minus_suffix','mean'))
morph_by_category=robust_sample.groupby('category').agg(
    n=('exact_recomputed','size'),mean_exact=('exact_recomputed','mean'),
    mean_suffix=('suffix_normalized','mean'), mean_drop=('drop_exact_minus_suffix','mean'))
morph_summary.to_csv(review_dir/'morph_robustness_summary.csv',index=False)
morph_by_source.to_csv(review_dir/'morph_robustness_by_source.csv')
morph_by_category.to_csv(review_dir/'morph_robustness_by_category.csv')
print('Paired suffix-robustness summary:')
display(morph_summary)
print('By source:')
display(morph_by_source)

# Journal-ready figure with consistent labels; only aggregate values.
fig,ax=plt.subplots(figsize=(5.4,3.35),dpi=160)
values=morph_by_source.sort_values('mean_exact')
y=np.arange(len(values))
ax.barh(y-.17, values['mean_exact'],height=.33,label='Exact overlap divergence')
ax.barh(y+.17, values['mean_suffix'],height=.33,label='Suffix-normalized divergence')
ax.set_yticks(y,values.index)
ax.set_xlabel('Mean headline--body divergence')
ax.legend(loc='lower right',fontsize=8)
fig.tight_layout()
fig.savefig(review_dir/'morph_robustness_by_source.pdf',bbox_inches='tight')
fig.savefig(review_dir/'morph_robustness_by_source.png',dpi=300,bbox_inches='tight')
plt.close(fig)

# 4. Ingestion provenance audit: local per-file counts; upstream published 664,880.
per_file=raw_df.groupBy('_file_path').count().orderBy('_file_path').toPandas()
per_file.to_csv(review_dir/'ingestion_rows_per_file.csv',index=False)
audit={'upstream_reported_raw_articles':664880,
       'current_ingested_rows':int(raw_df.count()),
       'difference_vs_upstream':int(raw_df.count()-664880),
       'n_csv_files':int(len(per_file)),
       'status':'Difference needs source-file/version verification; do not infer its cause.'}
(review_dir/'provenance_audit.json').write_text(json.dumps(audit,indent=2))


## Reviewer check: Weekday sensitivity


In [ ]:
# Reviewer check 4: seven-day seasonality sensitivity for total publication spikes.
# Prior eight SAME weekdays define a trailing 56-day comparable-day baseline.
# This deliberately checks robustness; it is not a newly validated event detector.
import pandas as pd,numpy as np,json
all_daily=daily.select('publication_date','daily_articles').toPandas()
all_daily['publication_date']=pd.to_datetime(all_daily['publication_date'])
all_daily=all_daily.sort_values('publication_date').set_index('publication_date')
all_daily['weekday']=all_daily.index.dayofweek
all_daily['weekday_name']=all_daily.index.day_name()
all_daily['weekday_mean8']=all_daily.groupby('weekday')['daily_articles'].transform(
    lambda s:s.shift(1).rolling(8,min_periods=6).mean())
all_daily['weekday_sd8']=all_daily.groupby('weekday')['daily_articles'].transform(
    lambda s:s.shift(1).rolling(8,min_periods=6).std(ddof=1))
all_daily['weekday_z']=(all_daily['daily_articles']-all_daily['weekday_mean8'])/all_daily['weekday_sd8'].replace(0,np.nan)
all_daily['weekday_spike']=all_daily['weekday_z']>=3
original_spikes={pd.Timestamp(r['publication_date']) for r in spikes.select('publication_date').collect()}
weekday_spikes=set(all_daily.index[all_daily['weekday_spike']])
original_vs_weekday=all_daily.loc[all_daily.index.isin(original_spikes)|all_daily['weekday_spike'],
                 ['daily_articles','weekday_name','weekday_mean8','weekday_sd8','weekday_z','weekday_spike']].copy()
original_vs_weekday['original_spike']=original_vs_weekday.index.isin(original_spikes)
original_vs_weekday.to_csv(review_dir/'weekday_spike_sensitivity.csv',index_label='date')
all_daily.groupby('weekday_name').daily_articles.agg(['count','mean','median','std']).to_csv(review_dir/'weekday_volume_profile.csv')
comparison={'original_28day_spike_n':len(original_spikes),'weekday_adjusted_spike_n':len(weekday_spikes),
            'overlap_n':len(original_spikes&weekday_spikes),
            'original_spikes_not_weekday_confirmed':[str(t.date()) for t in sorted(original_spikes-weekday_spikes)]}
(review_dir/'weekday_spike_summary.json').write_text(json.dumps(comparison,indent=2))
print(comparison)
display(original_vs_weekday)


## Reviewer check: Headline-length discretization


In [ ]:
# Reviewer check 5: quantify score discretization by headline length.
# A five-term headline can yield only multiples of 1/5 before rounding.
from pyspark.sql import functions as F
length_counts=divergence_df.groupBy('newspaper_source','n_headline_terms').count().toPandas()
length_counts.to_csv(review_dir/'headline_length_by_source.csv',index=False)
length_counts['five_term']=length_counts['n_headline_terms'].eq(5)
length_counts['six_or_less']=length_counts['n_headline_terms'].le(6)
length_counts['five_n']=length_counts['count']*length_counts['five_term']
length_counts['short_n']=length_counts['count']*length_counts['six_or_less']
length_summary=length_counts.groupby('newspaper_source').agg(
    article_n=('count','sum'),five_term_n=('five_n','sum'),
    six_or_less_n=('short_n','sum')).reset_index()
length_summary['five_term_pct']=100*length_summary['five_term_n']/length_summary['article_n']
length_summary['six_or_less_pct']=100*length_summary['six_or_less_n']/length_summary['article_n']
length_summary.to_csv(review_dir/'headline_length_discretization_summary.csv',index=False)
length_effect=(divergence_df.withColumn('length_bin',
               F.when(F.col('n_headline_terms')>=9,F.lit('9+'))
                .otherwise(F.col('n_headline_terms').cast('string')))
               .groupBy('newspaper_source','length_bin')
               .agg(F.count('*').alias('article_n'),
                    F.avg('headline_body_divergence').alias('mean_divergence'))
               .toPandas())
length_effect.to_csv(review_dir/'divergence_stratified_by_headline_length.csv',index=False)
display(length_summary.sort_values('five_term_pct',ascending=False))


In [ ]:

# Export final results for manuscript update.
# Run ONLY this cell, not the entire notebook.

manifest = {
    'original_nlp_sample': int(tokenized_df.count()),
    'original_lda_training': int(lda_train.count()),

    'lda_expanded': expanded_lda_pd.to_dict(orient='records'),

    'lag_tests_total': int(len(lag_fdr_pd)),

    'lag_global_selected_bh_confirmed': int(
        pair_summary.loc[
            pair_summary['category'] == 'Overall',
            'bh_confirmed_positive_lead'
        ].sum()
    ),

    'lag_fdr_summary': summary.to_dict(orient='records'),

    'morphology_robustness': morph_summary.iloc[0].to_dict(),

    'provenance': audit,
    'weekday_spike_summary': comparison,
    'headline_length_discretization': length_summary.to_dict(orient='records'),
}

output_path = review_dir / 'MANUSCRIPT_UPDATE_RESULTS.json'

output_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
        default=str
    ),
    encoding='utf-8'
)

print("Results successfully saved!")
print("File:", output_path)